# Phase 0 — Setup & Configuration

This notebook ingests the **23 United States Coast Guard Reports of Investigation (ROI)** in
`Data/raw/unstructured/us_coast_guard_reports/` (`<VESSEL>_<MISLE>_ROI_Redacted.pdf`, 580 pages) as **one source**:
a corpus of documents written on the same ROI template (plan §4.1 — one notebook, one extracted, one cleaned and one
chunks file, `filename` per chunk).

Their RAG value is (a) real marine-casualty narratives — what happened, why, and what the Coast Guard recommended — and
(b) an **IMO incident profile per report**, coded with the field catalogue of the IMO circular MSC-MEPC.3/Circ.4/Rev.1
(ADR-023), so later consumers can count or filter incidents by IMO category.

How the corpus is processed (plan §1, owner's cost rule): the format is **never learned by parsing every PDF**. It is
learned from a **seeded random sample** (seed 42 → VICAR_602, CUMBERLAND_II, BIG_STICK, KIM_LY) plus one image-only
report for the OCR path, written once as a reviewed template, then every report is **fingerprinted** cheaply against
it; one template-driven parser handles the conforming reports, and only the deviating ones are inspected individually.

The phase establishes:

- libraries and **one PDF access helper** (`page_dict(report, page)`) used by every later phase
- project / output directories, including the **OCR cache** owned by this notebook (`Data/processed/rag/ocr_cache/USCG_ROI/`)
- source metadata. Access: public, redacted federal reports → **level 1 / Operations for every unit** (`docs/rbac_mapping.md` §4),
  asserted per unit (Phase 9) and per chunk (Phase 15)
- the corpus file list with one SHA-256 fingerprint per file, and the seeded sample

Plan: `docs/plans/USCG_ROI_chunking_plan.md` (§4 decisions, §6 implementation log).

In [1]:
# =========================================================
# 0.1 Imports & Environment
# =========================================================

from pathlib import Path
from collections import Counter
import re
import json
import random
import hashlib
import unicodedata
import difflib

import pandas as pd
import numpy as np
import pymupdf

print("Imports loaded successfully.")
print("pandas :", pd.__version__)
print("numpy  :", np.__version__)
print("pymupdf:", pymupdf.__version__)

Imports loaded successfully.
pandas : 3.0.5
numpy  : 2.2.4
pymupdf: 1.28.2


In [2]:
# =========================================================
# 0.2 Locate the corpus folder and list the reports
# =========================================================
# File name = <VESSEL>_<MISLE activity number (7 digits)>_ROI_Redacted.pdf. report_id = the vessel part.

CORPUS_RELATIVE_PATH = Path("Data/raw/unstructured/us_coast_guard_reports")
FILE_RE = re.compile(r"^(?P<report_id>.+)_(?P<misle>\d{7})_ROI_Redacted\.pdf$")

search_roots = [Path.cwd(), *Path.cwd().parents]
candidates = [root / CORPUS_RELATIVE_PATH for root in search_roots]
CORPUS_DIR = next((path for path in candidates if path.is_dir()), None)
if CORPUS_DIR is None:
    raise FileNotFoundError("USCG corpus folder not found.\n\nLooked in:\n" + "\n".join(f"  {p}" for p in candidates))

CORPUS_FILES = sorted(p.name for p in CORPUS_DIR.glob("*.pdf"))
bad_names = [f for f in CORPUS_FILES if not FILE_RE.match(f)]
assert not bad_names, f"Unexpected file names: {bad_names}"

reports_df = pd.DataFrame([{**FILE_RE.match(f).groupdict(), "filename": f} for f in CORPUS_FILES])
REPORT_IDS = reports_df["report_id"].tolist()
FILE_OF = dict(zip(reports_df["report_id"], reports_df["filename"]))
MISLE_OF_FILE = dict(zip(reports_df["report_id"], reports_df["misle"]))

print("Corpus folder:", CORPUS_DIR)
print("Reports      :", len(REPORT_IDS))
print(reports_df.to_string(index=False))

Corpus folder: C:\Users\terbo\Desktop\Project\Data\raw\unstructured\us_coast_guard_reports
Reports      : 23
         report_id   misle                                    filename
         BIG_STICK 7895450          BIG_STICK_7895450_ROI_Redacted.pdf
         CAROL_ANN 7816410          CAROL_ANN_7816410_ROI_Redacted.pdf
           CONCHRD 7107811            CONCHRD_7107811_ROI_Redacted.pdf
     CUMBERLAND_II 8126294      CUMBERLAND_II_8126294_ROI_Redacted.pdf
          CUYAHOGA 7881082           CUYAHOGA_7881082_ROI_Redacted.pdf
 GENERAL_MACARTHUR 7861135  GENERAL_MACARTHUR_7861135_ROI_Redacted.pdf
         GOOD_GULF 8194633          GOOD_GULF_8194633_ROI_Redacted.pdf
         H20_BELOW 6966090          H20_BELOW_6966090_ROI_Redacted.pdf
            KIM_LY 7867677             KIM_LY_7867677_ROI_Redacted.pdf
          LA6893ER 8027803           LA6893ER_8027803_ROI_Redacted.pdf
        LADY_ELLEN 7965695         LADY_ELLEN_7965695_ROI_Redacted.pdf
          LITTLE_G 7717011           LI

In [3]:
# =========================================================
# 0.3 Open the PDFs + PDF access helpers
# =========================================================
# Every phase reads native text through page_dict(), so block/line indexes are identical everywhere.
# TEXT_FLAGS = the default "dict" flags WITHOUT image bytes (images are inventoried separately in Phase 1).

TEXT_FLAGS = pymupdf.TEXTFLAGS_DICT & ~pymupdf.TEXT_PRESERVE_IMAGES

docs: dict[str, pymupdf.Document] = {rid: pymupdf.open(CORPUS_DIR / FILE_OF[rid]) for rid in REPORT_IDS}
_dict_cache: dict[tuple[str, int], dict] = {}


def page_dict(report_id: str, p: int) -> dict:
    """get_text('dict') of physical page p (1-based) of one report, text blocks only, cached."""
    key = (report_id, p)
    if key not in _dict_cache:
        _dict_cache[key] = docs[report_id][p - 1].get_text("dict", flags=TEXT_FLAGS)
    return _dict_cache[key]


print("Documents opened:", len(docs))
print("Pages per report:", {rid: len(d) for rid, d in docs.items()})

Documents opened: 23
Pages per report: {'BIG_STICK': 18, 'CAROL_ANN': 51, 'CONCHRD': 45, 'CUMBERLAND_II': 13, 'CUYAHOGA': 33, 'GENERAL_MACARTHUR': 22, 'GOOD_GULF': 11, 'H20_BELOW': 16, 'KIM_LY': 15, 'LA6893ER': 19, 'LADY_ELLEN': 19, 'LITTLE_G': 17, 'MERRICK': 19, 'MICHIPICOTEN': 21, 'NC3250BW': 15, 'PROWLER_ATTESSA_IV': 22, 'QUEEN_OF_THE_WEST': 23, 'SPIRIT_OF_BOSTON': 97, 'TOO_SHORT': 16, 'UNA_MAS': 14, 'VICAR_602': 17, 'WAYMON_BOYD': 38, 'WILDERNESS_I': 19}


In [4]:
# =========================================================
# 0.4 Inspect PDF metadata (producer tells how each file was made)
# =========================================================
# Word -> PDF (Distiller / PDFMaker / "Microsoft: Print To PDF"), a scanner (Lexmark), Acrobat "Paper Capture" (OCR).
# The embedded title/author are not used: title, vessel and dates are read from the reports themselves (Phase 4).

metadata_df = pd.DataFrame([
    {"report_id": rid, "pages": len(d), "producer": d.metadata.get("producer", ""),
     "creator": d.metadata.get("creator", ""), "toc_entries": len(d.get_toc())}
    for rid, d in docs.items()
])
with pd.option_context("display.max_colwidth", 50):
    display(metadata_df)

,report_id,pages,producer,creator,toc_entries
0,BIG_STICK,18,Acrobat Distiller 25.0 (Windows),Acrobat PDFMaker 25 for Word,4
1,CAROL_ANN,51,Microsoft: Print To PDF,,0
2,CONCHRD,45,Adobe PDF Library 25.1.201,Acrobat PDFMaker 25 for Word,0
3,CUMBERLAND_II,13,,,0
4,CUYAHOGA,33,,,0
5,GENERAL_MACARTHUR,22,,,0
6,GOOD_GULF,11,,,0
7,H20_BELOW,16,,,0
8,KIM_LY,15,Microsoft: Print To PDF,,0
9,LA6893ER,19,Acrobat Distiller 25.0 (Windows),PScript5.dll Version 5.2.2,0


In [5]:
# =========================================================
# 0.5 Define project and output paths
# =========================================================

PROJECT_ROOT = CORPUS_DIR.parents[3]                          # <root>/Data/raw/unstructured/us_coast_guard_reports
assert (PROJECT_ROOT / "pyproject.toml").is_file(), f"Project root not found: {PROJECT_ROOT}"

PROCESSED_RAG_DIR = PROJECT_ROOT / "Data" / "processed" / "rag"
EXTRACTED_DIR = PROCESSED_RAG_DIR / "extracted"
CLEANED_DIR = PROCESSED_RAG_DIR / "cleaned"
CHUNKS_DIR = PROCESSED_RAG_DIR / "chunks"
OCR_CACHE_DIR = PROCESSED_RAG_DIR / "ocr_cache" / "USCG_ROI"   # owned by this notebook: one JSON per OCR'd page
SCRATCH_DIR = PROJECT_ROOT / "Notebooks" / "_scratch"          # git-ignored review aids only

for directory in (EXTRACTED_DIR, CLEANED_DIR, CHUNKS_DIR, OCR_CACHE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("Project root  :", PROJECT_ROOT)
print("Extracted dir :", EXTRACTED_DIR)
print("Cleaned dir   :", CLEANED_DIR)
print("Chunks dir    :", CHUNKS_DIR)
print("OCR cache dir :", OCR_CACHE_DIR)
print("Scratch dir   :", SCRATCH_DIR)

Project root  : C:\Users\terbo\Desktop\Project
Extracted dir : C:\Users\terbo\Desktop\Project\Data\processed\rag\extracted
Cleaned dir   : C:\Users\terbo\Desktop\Project\Data\processed\rag\cleaned
Chunks dir    : C:\Users\terbo\Desktop\Project\Data\processed\rag\chunks
OCR cache dir : C:\Users\terbo\Desktop\Project\Data\processed\rag\ocr_cache\USCG_ROI
Scratch dir   : C:\Users\terbo\Desktop\Project\Notebooks\_scratch


In [6]:
# =========================================================
# 0.6 Define source and pipeline configuration
# =========================================================

SOURCE_NAME = "USCG_ROI"
SOURCE_PUBLISHER = "United States Coast Guard"
DOCUMENT_TYPE = "uscg_report_of_investigation"
REAL_OR_SYNTHETIC = "real"
# version_date is per report = date of the Commandant's action (plan §4.8), read in Phase 4.

# External, publicly released (redacted) reports -> level 1 for everyone, owner = Operations (docs/rbac_mapping.md §4).
DOCUMENT_ACCESS = {"sensitivity_level": 1, "department": "Operations"}
KNOWN_DEPARTMENTS = {"Operations", "Customs", "Finance", "HR", "IT"}   # rows of the departments table

SOURCE_DESCRIPTION = ("United States Coast Guard Reports of Investigation (ROI) into 23 marine casualties — "
                      "public redacted releases written on one shared ROI template")

MIN_CHUNK_CHARS = 300                  # a guideline here, not a rule (plan §1.3): the split follows the content
MAX_CHUNK_CHARS = 2000                 # safety cap
FRAGMENT_MIN_CHARS = 50                # below this a chunk is a fragment with no standalone meaning (forbidden)
CHUNK_PREFIX = "uscg"

source_metadata = {
    "source_name": SOURCE_NAME,
    "publisher": SOURCE_PUBLISHER,
    "document_type": DOCUMENT_TYPE,
    "real_or_synthetic": REAL_OR_SYNTHETIC,
    "files": len(REPORT_IDS),
    "access_policy": "document_level_asserted_per_unit",
    "access": DOCUMENT_ACCESS,
    "description": SOURCE_DESCRIPTION,
}

print("Source configuration:")
for key, value in source_metadata.items():
    print(f"{key:20}: {value}")
print("\nChunk configuration:")
print("MIN_CHUNK_CHARS    :", MIN_CHUNK_CHARS, "(guideline)")
print("MAX_CHUNK_CHARS    :", MAX_CHUNK_CHARS, "(safety cap)")
print("FRAGMENT_MIN_CHARS :", FRAGMENT_MIN_CHARS)
print("CHUNK_PREFIX       :", CHUNK_PREFIX)

Source configuration:
source_name         : USCG_ROI
publisher           : United States Coast Guard
document_type       : uscg_report_of_investigation
real_or_synthetic   : real
files               : 23
access_policy       : document_level_asserted_per_unit
access              : {'sensitivity_level': 1, 'department': 'Operations'}
description         : United States Coast Guard Reports of Investigation (ROI) into 23 marine casualties — public redacted releases written on one shared ROI template

Chunk configuration:
MIN_CHUNK_CHARS    : 300 (guideline)
MAX_CHUNK_CHARS    : 2000 (safety cap)
FRAGMENT_MIN_CHARS : 50
CHUNK_PREFIX       : uscg


In [7]:
# =========================================================
# 0.7 Source file fingerprints (one SHA-256 per report)
# =========================================================

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    sha256 = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            sha256.update(chunk)
    return sha256.hexdigest()


reports_df["sha256"] = reports_df["filename"].map(lambda f: sha256_file(CORPUS_DIR / f))
reports_df["file_size"] = reports_df["filename"].map(lambda f: (CORPUS_DIR / f).stat().st_size)
reports_df["pages"] = reports_df["report_id"].map(lambda rid: len(docs[rid]))
SHA_OF = dict(zip(reports_df["report_id"], reports_df["sha256"]))

display(reports_df[["report_id", "misle", "pages", "file_size", "sha256"]])
print(f"Total: {reports_df['pages'].sum()} pages, {reports_df['file_size'].sum():,} bytes")

,report_id,misle,pages,file_size,sha256
0,BIG_STICK,7895450,18,4696740,63197d4ec3abcb20830669b54b6388dbee1463680e37b0...
1,CAROL_ANN,7816410,51,2511485,d2435424f7e9e420c2cb2529961f8269de7855def7581f...
2,CONCHRD,7107811,45,3579950,67762d08caf4f265a4f6b001711884e261fefb10cb629c...
3,CUMBERLAND_II,8126294,13,1921809,8794152c28321eb2e4a90e86b339751332ecf2a6c5ceb8...
4,CUYAHOGA,7881082,33,4631257,c518b7d648f90ab65608426f5a39cb32667954bc1cb689...
5,GENERAL_MACARTHUR,7861135,22,2788630,7a5f54a53ec5bce4939adadd6fe89ac91483ca57e94928...
6,GOOD_GULF,8194633,11,2019382,b079d8d08db1276fde6ebd4189b6f251d8ab81bf1fbd74...
7,H20_BELOW,6966090,16,2181980,f3fcf7689854c086f79383529c2e174af10699e91199f1...
8,KIM_LY,7867677,15,7497021,070b37ee21b7addd2d4e5bb6e73f471703fe5b4f25957f...
9,LA6893ER,8027803,19,1804294,401df0fb942bc458c76d1303d640c9431b1c9cb48c0b9a...


Total: 580 pages, 100,188,376 bytes


In [8]:
# =========================================================
# 0.8 Seeded random sample (the template is learned from these reports only)
# =========================================================
# random.sample over the sorted file list with seed 42 (plan §0/§4.9). One image-only report is added for the OCR
# path: the one with the fewest pages among the four image-only bodies (plan §0), chosen by page count — UNA_MAS.

random.seed(42)
SAMPLE_FILES = random.sample(CORPUS_FILES, 4)
SAMPLE_REPORTS = [FILE_RE.match(f).group("report_id") for f in SAMPLE_FILES]

IMAGE_ONLY_BODY_REPORTS = ["H20_BELOW", "PROWLER_ATTESSA_IV", "QUEEN_OF_THE_WEST", "UNA_MAS"]   # plan §0, checked in Phase 1
OCR_SAMPLE_REPORT = min(IMAGE_ONLY_BODY_REPORTS, key=lambda rid: len(docs[rid]))

print("Seeded sample (seed 42):", SAMPLE_REPORTS)
print("OCR sample report      :", OCR_SAMPLE_REPORT, f"({len(docs[OCR_SAMPLE_REPORT])} pages)")

Seeded sample (seed 42): ['VICAR_602', 'CUMBERLAND_II', 'BIG_STICK', 'KIM_LY']
OCR sample report      : UNA_MAS (14 pages)


In [9]:
# =========================================================
# 0.9 Phase 0 validation
# =========================================================

assert len(REPORT_IDS) == 23, f"ADR-007: 23 reports expected, found {len(REPORT_IDS)}."
assert reports_df["report_id"].is_unique and reports_df["misle"].is_unique
assert reports_df["pages"].sum() == 580, "Reconnaissance (plan §0): 580 pages."
assert reports_df["sha256"].str.len().eq(64).all() and reports_df["sha256"].is_unique
assert (reports_df["file_size"] > 0).all()

assert DOCUMENT_TYPE == "uscg_report_of_investigation" and REAL_OR_SYNTHETIC == "real"
assert DOCUMENT_ACCESS == {"sensitivity_level": 1, "department": "Operations"}, "rbac_mapping §4: USCG = level 1 / Operations"
assert DOCUMENT_ACCESS["department"] in KNOWN_DEPARTMENTS
assert 0 < FRAGMENT_MIN_CHARS < MIN_CHUNK_CHARS < MAX_CHUNK_CHARS

# ---- reviewed snapshot (2026-09-26) ----
assert SAMPLE_REPORTS == ["VICAR_602", "CUMBERLAND_II", "BIG_STICK", "KIM_LY"], SAMPLE_REPORTS
assert OCR_SAMPLE_REPORT == "UNA_MAS"
assert not any(b.get("type") == 1 for b in page_dict("BIG_STICK", 2)["blocks"]), "page_dict must not contain image blocks."

print("✅ Phase 0 validation passed.")
print("Reports:", len(REPORT_IDS), "| pages:", int(reports_df["pages"].sum()), "| sample:", SAMPLE_REPORTS, "+", OCR_SAMPLE_REPORT)

✅ Phase 0 validation passed.
Reports: 23 | pages: 580 | sample: ['VICAR_602', 'CUMBERLAND_II', 'BIG_STICK', 'KIM_LY'] + UNA_MAS


# Phase 1 — Corpus Inventory (all 23 reports, cheap)

One row per physical page of every report, from cheap PyMuPDF calls only (no layout parsing): characters and words
of the text layer, image placements and the share of the page covered by the largest image, and a **text trace**
summary — how many characters are drawn invisibly (render mode 3, the classic OCR layer) and how many use Acrobat's
OCR-substitute fonts (names starting with `*`, e.g. `*Minion Pro-Bold-7757`, written by Acrobat when a scanned page
is converted to editable text).

Each page then receives a **text class**:

| Class | Rule | What happens next |
|---|---|---|
| `text` | a native text layer, not lying over a full-page scan | parsed directly |
| `ocr_layer` | text lying over a full-page image (invisible mode-3 layer, or Acrobat editable OCR text) | readability gate (Phase 1b): kept if it passes, re-OCR'd otherwise |
| `image_only` | fewer than 20 visible characters | OCR'd with RapidOCR (Phase 1b); a page with no text boxes is a photo / graphic |

Reconnaissance correction (measured here): VICAR_602 is mostly native Word text — only its two scanned letters
(p.3, p.5) carry Acrobat OCR text; the same kind of scanned letter appears in MERRICK (p.4, p.6), CONCHRD (p.3) and
TOO_SHORT (p.4). MERRICK's body (p.7–19) is an invisible Paper-Capture layer. QUEEN_OF_THE_WEST p.22 has an
invisible layer made only of spaces → `image_only`.

In [10]:
# =========================================================
# 1.1 Build the page inventory of all reports
# =========================================================
MIN_TEXT_CHARS = 20            # below this a page has no usable text layer
FULL_PAGE_IMAGE = 0.90         # an image covering >= 90 % of the page = a scanned page

page_records = []
for rid in REPORT_IDS:
    doc = docs[rid]
    for page in doc:
        p = page.number + 1
        text = page.get_text("text") or ""
        stripped = text.strip()
        area = page.rect.width * page.rect.height
        infos = page.get_image_info()
        cover = max([(i["bbox"][2] - i["bbox"][0]) * (i["bbox"][3] - i["bbox"][1]) / area for i in infos] or [0.0])
        invisible = star_font = visible = 0
        for tr in page.get_texttrace():
            n = sum(1 for c in tr["chars"] if chr(c[0]).strip())      # non-blank characters only
            if tr["type"] == 3:
                invisible += n
            else:
                visible += n
                if tr["font"].startswith("*"):
                    star_font += n
        page_records.append({
            "report_id": rid, "page_number": p,
            "char_count": len(stripped), "word_count": len(stripped.split()),
            "image_placements": len(infos), "max_image_cover": round(min(cover, 1.0), 3),
            "visible_chars": visible, "invisible_chars": invisible, "star_font_chars": star_font,
            "page_width": round(page.rect.width, 1), "page_height": round(page.rect.height, 1),
            "text": text,
        })

pages_df = pd.DataFrame(page_records)
print("Page inventory created:", len(pages_df), "pages of", pages_df["report_id"].nunique(), "reports")
print("Columns:", list(pages_df.columns))

Page inventory created: 580 pages of 23 reports
Columns: ['report_id', 'page_number', 'char_count', 'word_count', 'image_placements', 'max_image_cover', 'visible_chars', 'invisible_chars', 'star_font_chars', 'page_width', 'page_height', 'text']


In [11]:
# =========================================================
# 1.2 Page text classes
# =========================================================

def text_class(r) -> tuple[str, str]:
    if r.char_count < MIN_TEXT_CHARS:
        why = "no text layer" if r.invisible_chars == 0 else "invisible layer without words (blank characters only)"
        return "image_only", f"{r.char_count} visible chars; {why}"
    if r.max_image_cover >= FULL_PAGE_IMAGE:
        kind = "invisible mode-3 OCR layer" if r.invisible_chars > r.visible_chars else "Acrobat editable OCR text (*fonts)"
        return "ocr_layer", f"text over a full-page scan ({kind})"
    return "text", "native text layer"


cls = [text_class(r) for r in pages_df.itertuples()]
pages_df["text_class"] = [c[0] for c in cls]
pages_df["class_reason"] = [c[1] for c in cls]

summary_df = (pages_df.pivot_table(index="report_id", columns="text_class", values="page_number", aggfunc="count",
                                   fill_value=0)
              .reindex(columns=["text", "ocr_layer", "image_only"], fill_value=0))
summary_df["pages"] = summary_df.sum(axis=1)
summary_df["image_only_pages"] = pages_df[pages_df["text_class"] == "image_only"].groupby("report_id")["page_number"].agg(list)
summary_df["ocr_layer_pages"] = pages_df[pages_df["text_class"] == "ocr_layer"].groupby("report_id")["page_number"].agg(list)
with pd.option_context("display.max_rows", None, "display.max_colwidth", 80):
    display(summary_df)
print("Class counts:", pages_df["text_class"].value_counts().to_dict())

text_class,text,ocr_layer,image_only,pages,image_only_pages,ocr_layer_pages
report_id,,,,,,
BIG_STICK,18,0,0,18,NaN,NaN
CAROL_ANN,51,0,0,51,NaN,NaN
CONCHRD,43,1,1,45,[4],[3]
CUMBERLAND_II,11,0,2,13,"[1, 4]",NaN
CUYAHOGA,27,0,6,33,"[1, 4, 5, 11, 20, 21]",NaN
GENERAL_MACARTHUR,20,0,2,22,"[3, 4]",NaN
GOOD_GULF,10,0,1,11,[8],NaN
H20_BELOW,2,0,14,16,"[1, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]",NaN
KIM_LY,11,0,4,15,"[4, 5, 6, 7]",NaN


Class counts: {'text': 446, 'image_only': 115, 'ocr_layer': 19}


In [12]:
# =========================================================
# 1.3 Inventory statistics + the OCR-layer pages
# =========================================================
print("Page sizes (width x height):")
print(pages_df.groupby(["page_width", "page_height"]).size().to_string())
print("\nCharacters per class:")
print(pages_df.groupby("text_class")[["char_count", "word_count"]].sum().to_string())

ocr_layer_df = pages_df[pages_df["text_class"] == "ocr_layer"]
with pd.option_context("display.max_rows", None, "display.max_colwidth", 70):
    display(ocr_layer_df[["report_id", "page_number", "char_count", "visible_chars", "invisible_chars",
                          "star_font_chars", "max_image_cover", "class_reason"]])

Page sizes (width x height):
page_width  page_height
596.0       842.0            2
608.0       790.1            3
610.1       790.1           13
612.0       790.1            3
            792.0          558
            792.2            1

Characters per class:
            char_count  word_count
text_class                        
image_only           0           0
ocr_layer        40355        6340
text            918158      141678


,report_id,page_number,char_count,visible_chars,invisible_chars,star_font_chars,max_image_cover,class_reason
71,CONCHRD,3,1862,1563,0,1563,1.0,text over a full-page scan (Acrobat editable OCR text (*fonts))
282,MERRICK,4,1668,1408,0,1408,1.0,text over a full-page scan (Acrobat editable OCR text (*fonts))
284,MERRICK,6,1177,980,0,980,1.0,text over a full-page scan (Acrobat editable OCR text (*fonts))
285,MERRICK,7,892,0,724,0,1.0,text over a full-page scan (invisible mode-3 OCR layer)
286,MERRICK,8,1792,0,1470,0,1.0,text over a full-page scan (invisible mode-3 OCR layer)
287,MERRICK,9,2763,0,2248,0,1.0,text over a full-page scan (invisible mode-3 OCR layer)
288,MERRICK,10,3399,0,2830,0,1.0,text over a full-page scan (invisible mode-3 OCR layer)
289,MERRICK,11,2730,0,2261,0,1.0,text over a full-page scan (invisible mode-3 OCR layer)
290,MERRICK,12,979,0,809,0,1.0,text over a full-page scan (invisible mode-3 OCR layer)
291,MERRICK,13,3551,0,2901,0,1.0,text over a full-page scan (invisible mode-3 OCR layer)


In [13]:
# =========================================================
# 1.4 Phase 1 validation
# =========================================================
assert len(pages_df) == 580 and pages_df.groupby("report_id").size().to_dict() == reports_df.set_index("report_id")["pages"].to_dict()
for rid, grp in pages_df.groupby("report_id"):
    assert grp["page_number"].tolist() == list(range(1, len(docs[rid]) + 1)), f"{rid}: page numbers not 1..N"
assert pages_df["text_class"].isin({"text", "ocr_layer", "image_only"}).all()
assert pages_df["class_reason"].str.len().gt(0).all()
# a scanned page with text must be recognisably OCR text (invisible layer or Acrobat *fonts), never a native layer
ol = pages_df[pages_df["text_class"] == "ocr_layer"]
assert ((ol["invisible_chars"] > ol["visible_chars"]) | (ol["star_font_chars"] > 0.5 * ol["visible_chars"])).all()

# ---- reviewed snapshot (2026-09-26): image-only and OCR-layer pages per report ----
REVIEWED_IMAGE_ONLY = {
    "CONCHRD": [4], "CUMBERLAND_II": [1, 4], "CUYAHOGA": [1, 4, 5, 11, 20, 21], "GENERAL_MACARTHUR": [3, 4],
    "GOOD_GULF": [8], "H20_BELOW": [1, *range(4, 17)], "KIM_LY": [4, 5, 6, 7], "LADY_ELLEN": [5, 6, 7, 8, 9],
    "LITTLE_G": [4, 5, 7, 9, 10, 11, 13, 15], "MERRICK": [5], "NC3250BW": [3],
    "PROWLER_ATTESSA_IV": [1, *range(4, 23)], "QUEEN_OF_THE_WEST": [1, *range(5, 24)],
    "SPIRIT_OF_BOSTON": [6, 7, 8, 9], "TOO_SHORT": [3], "UNA_MAS": [1, *range(5, 15)], "VICAR_602": [1, 4],
    "WAYMON_BOYD": [4, 5, 6, 7, 8, 9, 10], "WILDERNESS_I": [8, 9, 10, 11, 13],
}
REVIEWED_OCR_LAYER = {"CONCHRD": [3], "MERRICK": [4, 6, *range(7, 20)], "TOO_SHORT": [4], "VICAR_602": [3, 5]}

actual_io = pages_df[pages_df["text_class"] == "image_only"].groupby("report_id")["page_number"].agg(list).to_dict()
actual_ol = ol.groupby("report_id")["page_number"].agg(list).to_dict()
assert actual_io == REVIEWED_IMAGE_ONLY, {k: v for k, v in actual_io.items() if REVIEWED_IMAGE_ONLY.get(k) != v}
assert actual_ol == REVIEWED_OCR_LAYER, actual_ol
# plan §0: the four image-only bodies
for rid in IMAGE_ONLY_BODY_REPORTS:
    assert summary_df.at[rid, "image_only"] >= 0.6 * summary_df.at[rid, "pages"], rid

print("✅ Phase 1 validation passed.")
print("Pages:", len(pages_df), "| classes:", pages_df["text_class"].value_counts().to_dict())
print("Image-only pages:", sum(map(len, REVIEWED_IMAGE_ONLY.values())), "in", len(REVIEWED_IMAGE_ONLY), "reports")
print("OCR-layer pages :", sum(map(len, REVIEWED_OCR_LAYER.values())), "in", len(REVIEWED_OCR_LAYER), "reports")

✅ Phase 1 validation passed.
Pages: 580 | classes: {'text': 446, 'image_only': 115, 'ocr_layer': 19}
Image-only pages: 115 in 19 reports
OCR-layer pages : 19 in 4 reports


# Phase 1b — OCR of the pages without a usable text layer (RapidOCR)

Owner decision (plan §4.2): image-only pages are **OCR'd, not skipped**, with RapidOCR (`rapidocr` + `onnxruntime`,
installed with uv, models bundled in the package — no system install, no download).

**Method**

1. Render the page with PyMuPDF at **300 dpi** (RGB, no alpha).
2. RapidOCR detects text-line boxes and recognises each line. Two settings are changed from the defaults, both on
   evidence from UNA_MAS p.5 (the OCR sample):
   - `text_score = 0` — the default (0.5) **silently drops** every line below 0.5 confidence; here every line is kept
     with its confidence and the readability gate decides;
   - `max_side_len = 2000` (default) is kept: 3400 was 40 % slower and not better (each setting garbled a different
     long line).
3. **Retry of weak lines:** a line whose confidence is below `OCR_RETRY_CONF` is recognised again from its own crop
   of the full-resolution render (recognition only). On UNA_MAS p.5 the line
   "ve - sn r s o s t s es" (0.742) becomes "vessel capsized, turning over port side first and began floating upside
   down. C-1 was thrown into" (0.996). The better reading is kept; the first one stays in the cache as evidence.
4. Boxes are converted to **PDF coordinates** (pixels × 72 / dpi) so OCR lines go through the same row / zone / template
   logic as native lines; each line keeps its **confidence** and the provenance `ocr`.
5. **Cache** (artifact of this notebook): one JSON per page in `Data/processed/rag/ocr_cache/USCG_ROI/`, with a key
   (file sha256, page, dpi, engine settings, rapidocr / onnxruntime versions, model file hashes). A cache file whose key
   differs is stale and is recomputed, so a re-run never OCRs again unless something changed.

All `image_only` pages are OCR'd (a photo is only known to be a photo once OCR finds no text on it), and the 19
`ocr_layer` pages too, so that their existing layer can be compared with an independent reading (1b.4).

In [14]:
# =========================================================
# 1b.1 OCR engine, settings and cache key
# =========================================================
import importlib.metadata as importlib_metadata
import logging
from rapidocr import RapidOCR

logging.getLogger("RapidOCR").setLevel(logging.CRITICAL)

OCR_DPI = 300
OCR_MAX_SIDE_LEN = 2000
OCR_RETRY_CONF = 0.95          # lines below this are recognised again from their full-resolution crop
OCR_CROP_PAD = 4               # pixels around a line crop
OCR_PARAMS = {"Global.text_score": 0.0, "Global.max_side_len": OCR_MAX_SIDE_LEN, "Global.log_level": "critical"}

ocr_engine = RapidOCR(params=OCR_PARAMS)
_model_dir = Path(ocr_engine.cfg.Global.model_root_dir)
OCR_ENGINE_INFO = {
    "rapidocr": importlib_metadata.version("rapidocr"),
    "onnxruntime": importlib_metadata.version("onnxruntime"),
    "det": f"{ocr_engine.cfg.Det.ocr_version.value} {ocr_engine.cfg.Det.model_type.value}",
    "cls": f"{ocr_engine.cfg.Cls.ocr_version.value} {ocr_engine.cfg.Cls.model_type.value}",
    "rec": f"{ocr_engine.cfg.Rec.ocr_version.value} {ocr_engine.cfg.Rec.model_type.value}",
    "model_files": {p.name: sha256_file(p)[:16] for p in sorted(_model_dir.glob("*.onnx"))},
}
OCR_SETTINGS = {"dpi": OCR_DPI, "colorspace": "RGB", "alpha": False, "params": OCR_PARAMS,
                "retry_conf": OCR_RETRY_CONF, "crop_pad": OCR_CROP_PAD}


def ocr_cache_key(rid: str, p: int) -> dict:
    return {"report_id": rid, "filename": FILE_OF[rid], "sha256": SHA_OF[rid], "page_number": p,
            "settings": OCR_SETTINGS, "engine": OCR_ENGINE_INFO}


def ocr_cache_path(rid: str, p: int) -> Path:
    return OCR_CACHE_DIR / f"{rid}_p{p:03d}.json"


print(json.dumps({"settings": OCR_SETTINGS, "engine": OCR_ENGINE_INFO}, indent=1))

{
 "settings": {
  "dpi": 300,
  "colorspace": "RGB",
  "alpha": false,
  "params": {
   "Global.text_score": 0.0,
   "Global.max_side_len": 2000,
   "Global.log_level": "critical"
  },
  "retry_conf": 0.95,
  "crop_pad": 4
 },
 "engine": {
  "rapidocr": "3.9.2",
  "onnxruntime": "1.30.0",
  "det": "PP-OCRv6 small",
  "cls": "PP-OCRv4 mobile",
  "rec": "PP-OCRv6 small",
  "model_files": {
   "ch_ppocr_mobile_v2.0_cls_mobile.onnx": "e47acedf663230f8",
   "PP-OCRv6_det_small.onnx": "090f04abcd9d9a74",
   "PP-OCRv6_rec_small.onnx": "6f327246b50388f3"
  }
 }
}


In [15]:
# =========================================================
# 1b.2 OCR one page (render -> detect + recognise -> retry weak lines -> PDF coordinates), cached
# =========================================================

def _render(rid: str, p: int) -> np.ndarray:
    pix = docs[rid][p - 1].get_pixmap(dpi=OCR_DPI, colorspace=pymupdf.csRGB, alpha=False)
    return np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, 3)


def _run_ocr(rid: str, p: int) -> dict:
    img = _render(rid, p)
    # the three flags are passed on EVERY call: RapidOCR keeps the flags of the previous call, so a rec-only retry
    # (1b.2, weak lines) would otherwise switch detection off for all later pages
    res = ocr_engine(img, use_det=True, use_cls=True, use_rec=True)
    scale = 72 / OCR_DPI
    lines = []
    # a page where detection finds no text box returns an output without boxes -> no lines (a photo / blank scan)
    boxes = getattr(res, "boxes", None)
    boxes = [] if boxes is None else boxes
    for box, text, conf in zip(boxes, getattr(res, "txts", None) or [], getattr(res, "scores", None) or []):
        b = np.asarray(box, dtype=float)
        px0, py0, px1, py1 = b[:, 0].min(), b[:, 1].min(), b[:, 0].max(), b[:, 1].max()
        line = {"bbox": [round(v * scale, 1) for v in (px0, py0, px1, py1)], "text": text, "conf": round(float(conf), 4),
                "retry": None}
        if conf < OCR_RETRY_CONF:
            y0, y1 = max(0, int(py0) - OCR_CROP_PAD), int(py1) + OCR_CROP_PAD
            x0, x1 = max(0, int(px0) - OCR_CROP_PAD), int(px1) + OCR_CROP_PAD
            rr = ocr_engine(np.ascontiguousarray(img[y0:y1, x0:x1]), use_det=False, use_cls=False, use_rec=True)
            if rr.txts:
                r_text, r_conf = rr.txts[0], float(rr.scores[0])
                line["retry"] = {"text": r_text, "conf": round(r_conf, 4), "first_text": text, "first_conf": line["conf"],
                                 "kept": "retry" if r_conf > conf else "first"}
                if r_conf > conf:
                    line["text"], line["conf"] = r_text, round(r_conf, 4)
        lines.append(line)
    lines.sort(key=lambda l: (l["bbox"][1], l["bbox"][0]))
    return {"image_px": [int(img.shape[1]), int(img.shape[0])], "lines": lines}


def ocr_page(rid: str, p: int) -> tuple[dict, str]:
    """OCR result of one page and where it came from ('cache' or 'ocr')."""
    path, key = ocr_cache_path(rid, p), ocr_cache_key(rid, p)
    if path.exists():
        cached = json.loads(path.read_text(encoding="utf-8"))
        if cached.get("key") == key:
            return cached, "cache"
    result = {"key": key, **_run_ocr(rid, p)}
    path.write_text(json.dumps(result, ensure_ascii=False, indent=1), encoding="utf-8")
    return result, "ocr"


print("OCR helpers ready. Cache:", OCR_CACHE_DIR.relative_to(PROJECT_ROOT))

OCR helpers ready. Cache: Data\processed\rag\ocr_cache\USCG_ROI


In [16]:
# =========================================================
# 1b.3 OCR every image_only and ocr_layer page (first run ~15 min; later runs read the cache)
# =========================================================
import time

OCR_TARGETS = pages_df.loc[pages_df["text_class"].isin(["image_only", "ocr_layer"]), ["report_id", "page_number",
                                                                                       "text_class"]]
ocr_results: dict[tuple[str, int], dict] = {}
ocr_origin = Counter()
t_start = time.time()
for k, r in enumerate(OCR_TARGETS.itertuples(), start=1):
    ocr_results[(r.report_id, r.page_number)], origin = ocr_page(r.report_id, r.page_number)
    ocr_origin[origin] += 1
    if origin == "ocr" and k % 10 == 0:
        print(f"  {k}/{len(OCR_TARGETS)} pages ({time.time() - t_start:.0f}s)")
print(f"OCR pages: {len(ocr_results)} | from cache: {ocr_origin['cache']} | newly OCR'd: {ocr_origin['ocr']} "
      f"| {time.time() - t_start:.0f}s")

OCR pages: 134 | from cache: 134 | newly OCR'd: 0 | 0s


### 1b.4–1b.7 Figures inside scanned pages, readability gate, OCR-layer check

**Measured on the OCR output (1b.3):** none of the 115 `image_only` pages is a bare photo — every one carries text:
scanned letters, scanned body pages, cover pages, and body pages that contain a **figure** (photo, nautical chart,
dive-computer screen). RapidOCR also reads the text *inside* those figures: chart soundings ("28 | 36 | 36 | 35",
241 lines on MERRICK p.8), screen labels, and even personal names written on a diagram (CUYAHOGA p.20–21).
That text is part of the image, not of the report's prose, so it must never reach the body text.

**Figure regions on a scanned page (1b.4).** A scanned page has no image placements to tell where a figure is, so
the regions are found in the page image itself: the page is rendered in grey, **every text box is masked out**, and
what ink remains is graphics (photos, chart lines, drawn table borders, the letterhead seal, signatures, redaction
boxes). The ink is dilated and split into connected regions; a region of at least 60 × 40 pt is kept and typed:

- `redaction` — a solid black box (mean grey ≤ 10, all ink on straight runs): the FOIA redactions over signatures;
- `table` — thin straight rules **on white paper**: ink density ≤ 0.25, ≥ 60 % of the ink on long horizontal / vertical
  runs, and ≤ 15 % light fill once the text is painted white (the vessel particulars and persons tables). The light-fill
  test was added after a Google-Earth map with boxed callouts (WILDERNESS_I p.9) passed the first two tests;
- `graphic` — everything else (photos: dense ink; charts and maps: sparse, curved ink or a light background fill).

**Figure bands.** A Word figure is a block across the text column, and one chart can come out as several ink regions
side by side (UNA_MAS p.8: three regions, 55 soundings between them). Graphic regions of at least 100 × 50 pt are
therefore merged by their vertical extent into **bands**; every line whose centre lies in a band is
`figure_internal` (never chunked; the figure is represented by its caption, ADR-020), except a caption line
(`Figure N …`). Smaller regions (the letterhead seal, a signature scribble) and tables / redactions never make a band.
The same test is applied to the lines of the OCR-layer pages.

**Readability gate (1b.5)** — per page, on the **body lines only** (figure-internal lines excluded):
share of dictionary words (words of ≥ 2 letters; dictionary = every word seen at least twice on the 446 native-text
pages of the corpus — no new dependency) ≥ 0.75, and mean recognition confidence ≥ 0.90 (RapidOCR pages). The share
threshold sits below the lowest page reviewed as fully readable: H20_BELOW p.15 (0.781, confidence 0.994) — that
report is scanned end to end, so its diving vocabulary (rebreather, counter-lung…) is absent from the native pages.
A page that fails would be `ocr_unreadable`, with its measures as evidence, never silently (none fails).

**OCR layers (1b.6).** The 19 `ocr_layer` pages keep their existing layer if it passes the same gate (plan §4.2); the
independent RapidOCR reading is compared word by word and kept as evidence (agreement 0.89–1.00: the layers are as
good as RapidOCR).

**OCR noise (1b.7).** A reading below 0.80 confidence of at most two characters is a mark, not text (reviewed on the
renders: the letterhead seal read as "3", a signature pen stroke read as "V"); it is recorded as `ocr_noise`.

In [17]:
# =========================================================
# 1b.4 Graphic regions of the scanned pages (text masked out) -> figure-internal lines
# =========================================================
from scipy import ndimage

GRAPHIC_DPI = 100
GRAPHIC_MIN_W, GRAPHIC_MIN_H, GRAPHIC_MIN_INK = 60, 40, 300     # pt, pt, pixels
TABLE_MAX_DENSITY, TABLE_MIN_STRAIGHT, TABLE_MAX_LIGHT = 0.25, 0.60, 0.15
REDACTION_MAX_GREY, REDACTION_MIN_STRAIGHT = 10, 0.98
CAPTION_RE = re.compile(r"^Fig(?:ure|\.)?\s*\d+", re.IGNORECASE)
_H_RUN, _V_RUN = np.ones((1, 40), bool), np.ones((40, 1), bool)


def graphic_regions(rid: str, p: int, boxes: list[list[float]]) -> list[dict]:
    """Regions of non-text ink on page p (boxes = text boxes in PDF points, masked out before the analysis)."""
    s = GRAPHIC_DPI / 72
    pix = docs[rid][p - 1].get_pixmap(dpi=GRAPHIC_DPI, colorspace=pymupdf.csGRAY)
    grey = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width).copy()
    for x0, y0, x1, y1 in boxes:                     # text boxes are painted white: only graphics remain
        grey[max(0, int(y0 * s) - 2):int(y1 * s) + 3, max(0, int(x0 * s) - 2):int(x1 * s) + 3] = 255
    ink = grey < 200
    straight = ndimage.binary_opening(ink, _H_RUN) | ndimage.binary_opening(ink, _V_RUN)
    lab, _ = ndimage.label(ndimage.binary_dilation(ink, iterations=4))
    regions = []
    for k, sl in enumerate(ndimage.find_objects(lab), start=1):
        mask = ink[sl] & (lab[sl] == k)
        n_ink = int(mask.sum())
        w, h = (sl[1].stop - sl[1].start) / s, (sl[0].stop - sl[0].start) / s
        if w < GRAPHIC_MIN_W or h < GRAPHIC_MIN_H or n_ink < GRAPHIC_MIN_INK:
            continue
        density = n_ink / mask.size
        straight_share = float((straight[sl] & mask).sum() / n_ink)
        mean_grey = float(grey[sl][mask].mean())
        box = grey[sl]
        light_share = float(((box >= 150) & (box < 235)).mean())      # light fill (map / photo background), text excluded
        if mean_grey <= REDACTION_MAX_GREY and straight_share >= REDACTION_MIN_STRAIGHT:
            kind = "redaction"          # a solid black box (FOIA redaction, e.g. over a signature): not a figure
        elif density <= TABLE_MAX_DENSITY and straight_share >= TABLE_MIN_STRAIGHT and light_share <= TABLE_MAX_LIGHT:
            kind = "table"              # thin straight rules on white paper
        else:
            kind = "graphic"
        regions.append({"bbox": [round(sl[1].start / s, 1), round(sl[0].start / s, 1), round(sl[1].stop / s, 1),
                                 round(sl[0].stop / s, 1)],
                        "kind": kind, "density": round(density, 3), "straight_share": round(straight_share, 3),
                        "mean_grey": round(mean_grey, 1), "light_share": round(light_share, 3)})
    return regions


BAND_MIN_W, BAND_MIN_H, BAND_JOIN_GAP = 100, 50, 12      # pt


def figure_bands(regions: list[dict]) -> list[tuple[float, float]]:
    """Vertical bands occupied by figures: a Word figure is a block across the text column, and one chart can come
    out as several ink regions side by side (UNA_MAS p.8), so large graphic regions are merged by their y-range.
    Small regions (the letterhead seal, a signature scribble) never make a band."""
    spans = sorted((r["bbox"][1], r["bbox"][3]) for r in regions if r["kind"] == "graphic"
                   and r["bbox"][2] - r["bbox"][0] >= BAND_MIN_W and r["bbox"][3] - r["bbox"][1] >= BAND_MIN_H)
    bands = []
    for y0, y1 in spans:
        if bands and y0 <= bands[-1][1] + BAND_JOIN_GAP:
            bands[-1] = (bands[-1][0], max(bands[-1][1], y1))
        else:
            bands.append((y0, y1))
    return bands


def in_band(bbox, bands) -> bool:
    cy = (bbox[1] + bbox[3]) / 2
    return any(y0 <= cy <= y1 for y0, y1 in bands)


def layer_lines(rid: str, p: int) -> list[dict]:
    """Lines of an existing (OCR) text layer, in the same shape as the RapidOCR lines."""
    out = []
    for b in page_dict(rid, p)["blocks"]:
        for l in b["lines"]:
            t = "".join(s["text"] for s in l["spans"]).strip()
            if t:
                out.append({"bbox": [round(v, 1) for v in l["bbox"]], "text": t, "conf": None, "retry": None})
    return out


scan_records, graphic_records, band_records = [], [], []
for (rid, p), res in ocr_results.items():
    tclass = pages_df.loc[(pages_df["report_id"] == rid) & (pages_df["page_number"] == p), "text_class"].iloc[0]
    for source, lines in (("rapidocr", res["lines"]), ("layer", layer_lines(rid, p) if tclass == "ocr_layer" else None)):
        if lines is None:
            continue
        regions = graphic_regions(rid, p, [l["bbox"] for l in lines])
        bands = figure_bands(regions)
        for r in regions:
            graphic_records.append({"report_id": rid, "page_number": p, "source": source, **r})
        for b in bands:
            band_records.append({"report_id": rid, "page_number": p, "source": source, "y0": b[0], "y1": b[1]})
        for i, l in enumerate(lines):
            fig = in_band(l["bbox"], bands) and not CAPTION_RE.match(l["text"])
            scan_records.append({"report_id": rid, "page_number": p, "text_class": tclass, "source": source,
                                 "line_no": i, "bbox": l["bbox"], "text": l["text"], "conf": l["conf"],
                                 "retried": l["retry"] is not None, "figure_internal": fig})

scan_lines_df = pd.DataFrame(scan_records)
graphics_df = pd.DataFrame(graphic_records)
figure_bands_df = pd.DataFrame(band_records)
print("Scanned-page lines:", len(scan_lines_df), scan_lines_df.groupby("source").size().to_dict())
print("Figure-internal lines:", scan_lines_df.groupby("source")["figure_internal"].sum().to_dict())
print("Graphic regions:", graphics_df.groupby(["source", "kind"]).size().to_dict())
fig_pages = scan_lines_df[scan_lines_df["figure_internal"]].groupby(["report_id", "page_number", "source"]).size()
with pd.option_context("display.max_rows", None):
    display(fig_pages.rename("figure_internal_lines"))

Scanned-page lines: 5697 {'layer': 693, 'rapidocr': 5004}
Figure-internal lines: {'layer': 21, 'rapidocr': 652}
Graphic regions: {('layer', 'graphic'): 9, ('layer', 'redaction'): 4, ('layer', 'table'): 5, ('rapidocr', 'graphic'): 109, ('rapidocr', 'redaction'): 26, ('rapidocr', 'table'): 24}


report_id           page_number  source  
CUMBERLAND_II       1            rapidocr      3
CUYAHOGA            1            rapidocr      3
                    20           rapidocr      5
                    21           rapidocr      8
H20_BELOW           1            rapidocr      3
                    10           rapidocr      2
                    12           rapidocr    115
LITTLE_G            9            rapidocr      1
                    10           rapidocr      2
                    13           rapidocr      4
MERRICK             8            rapidocr    217
                    12           rapidocr      3
                    16           layer        21
                                 rapidocr     36
PROWLER_ATTESSA_IV  1            rapidocr      3
                    9            rapidocr      1
                    16           rapidocr     21
                    17           rapidocr      2
                    19           rapidocr      1
QUEEN_OF_THE_WEST   1      

In [18]:
# =========================================================
# 1b.5 Readability gate (body lines only)
# =========================================================
GATE_MIN_DICT_SHARE = 0.75       # below the lowest page reviewed as fully readable (H20_BELOW p.15: 0.781, diving terms)
GATE_MIN_MEAN_CONF = 0.90
WORD_RE = re.compile(r"[A-Za-z]+")

_vocab_counts = Counter(w.lower() for t in pages_df.loc[pages_df["text_class"] == "text", "text"] for w in WORD_RE.findall(t))
OCR_VOCAB = {w for w, n in _vocab_counts.items() if n >= 2 and len(w) >= 2}


def gate_measures(lines: pd.DataFrame) -> dict:
    words = [w.lower() for t in lines["text"] for w in WORD_RE.findall(t) if len(w) >= 2]
    confs = lines["conf"].dropna()
    return {"body_lines": len(lines), "words": len(words),
            "dict_share": round(float(np.mean([w in OCR_VOCAB for w in words])), 3) if words else 0.0,
            "mean_word_len": round(float(np.mean([len(w) for w in words])), 2) if words else 0.0,
            "mean_conf": round(float(confs.mean()), 4) if len(confs) else None}


gate_records = []
for (rid, p, source), grp in scan_lines_df.groupby(["report_id", "page_number", "source"]):
    m = gate_measures(grp[~grp["figure_internal"]])
    passed = m["dict_share"] >= GATE_MIN_DICT_SHARE and (m["mean_conf"] is None or m["mean_conf"] >= GATE_MIN_MEAN_CONF)
    gate_records.append({"report_id": rid, "page_number": p, "source": source,
                         "text_class": grp["text_class"].iloc[0], **m,
                         "figure_internal_lines": int(grp["figure_internal"].sum()), "gate": "pass" if passed else "fail"})
gate_df = pd.DataFrame(gate_records)

print("Dictionary:", len(OCR_VOCAB), "words | thresholds: dict_share >=", GATE_MIN_DICT_SHARE, "| mean_conf >=", GATE_MIN_MEAN_CONF)
print("Gate:", gate_df.groupby(["source", "gate"]).size().to_dict())
with pd.option_context("display.max_rows", None):
    display(gate_df.sort_values("dict_share").head(12))

Dictionary: 4857 words | thresholds: dict_share >= 0.75 | mean_conf >= 0.9
Gate: {('layer', 'pass'): 19, ('rapidocr', 'pass'): 134}


,report_id,page_number,source,text_class,body_lines,words,dict_share,mean_word_len,mean_conf,figure_internal_lines,gate
26,H20_BELOW,15,rapidocr,image_only,41,474,0.781,5.39,0.9942,0,pass
52,MERRICK,8,layer,ocr_layer,24,278,0.874,4.86,NaN,0,pass
53,MERRICK,8,rapidocr,ocr_layer,24,278,0.881,4.87,0.9949,217,pass
84,PROWLER_ATTESSA_IV,10,rapidocr,image_only,50,117,0.889,5.67,0.9948,0,pass
106,QUEEN_OF_THE_WEST,13,rapidocr,image_only,25,277,0.895,5.53,0.9927,11,pass
21,H20_BELOW,10,rapidocr,image_only,26,239,0.895,5.23,0.9926,2,pass
104,QUEEN_OF_THE_WEST,11,rapidocr,image_only,34,290,0.897,4.78,0.9924,0,pass
132,UNA_MAS,12,rapidocr,image_only,44,484,0.897,5.67,0.9941,0,pass
101,QUEEN_OF_THE_WEST,8,rapidocr,image_only,53,591,0.898,4.87,0.9924,0,pass
131,UNA_MAS,11,rapidocr,image_only,42,448,0.902,5.41,0.9944,0,pass


In [19]:
# =========================================================
# 1b.6 OCR layers: gate + word agreement with the independent RapidOCR reading
# =========================================================

def word_agreement(a: str, b: str) -> float:
    wa, wb = WORD_RE.findall(a.lower()), WORD_RE.findall(b.lower())
    return round(difflib.SequenceMatcher(None, wa, wb, autojunk=False).ratio(), 3)


layer_check = []
for (rid, p), grp in scan_lines_df[scan_lines_df["text_class"] == "ocr_layer"].groupby(["report_id", "page_number"]):
    body = grp[~grp["figure_internal"]]
    txt = {s: "\n".join(body.loc[body["source"] == s, "text"]) for s in ("layer", "rapidocr")}
    g = gate_df.set_index(["report_id", "page_number", "source"])
    layer_check.append({"report_id": rid, "page_number": p,
                        "layer_dict_share": g.at[(rid, p, "layer"), "dict_share"],
                        "rapidocr_dict_share": g.at[(rid, p, "rapidocr"), "dict_share"],
                        "word_agreement": word_agreement(txt["layer"], txt["rapidocr"]),
                        "layer_gate": g.at[(rid, p, "layer"), "gate"]})
layer_check_df = pd.DataFrame(layer_check)
layer_check_df["decision"] = np.where(layer_check_df["layer_gate"] == "pass", "keep_layer", "re_ocr")
display(layer_check_df)

,report_id,page_number,layer_dict_share,rapidocr_dict_share,word_agreement,layer_gate,decision
0,CONCHRD,3,0.972,0.972,0.949,pass,keep_layer
1,MERRICK,4,0.963,0.968,0.930,pass,keep_layer
2,MERRICK,6,0.974,0.974,0.922,pass,keep_layer
3,MERRICK,7,0.912,0.912,0.918,pass,keep_layer
4,MERRICK,8,0.874,0.881,0.983,pass,keep_layer
5,MERRICK,9,0.958,0.965,0.994,pass,keep_layer
6,MERRICK,10,0.958,0.960,0.996,pass,keep_layer
7,MERRICK,11,0.947,0.947,1.000,pass,keep_layer
8,MERRICK,12,0.919,0.917,0.967,pass,keep_layer
9,MERRICK,13,0.916,0.916,0.998,pass,keep_layer


In [20]:
# =========================================================
# 1b.7 Text source of every page + weak OCR lines
# =========================================================
# text source per page: native layer (text), existing OCR layer (ocr_layer, gate passed) or RapidOCR (image_only)
source_of = {}
for r in pages_df.itertuples():
    if r.text_class == "text":
        source_of[(r.report_id, r.page_number)] = "native"
    elif r.text_class == "ocr_layer":
        keep = layer_check_df.set_index(["report_id", "page_number"]).at[(r.report_id, r.page_number), "decision"] == "keep_layer"
        source_of[(r.report_id, r.page_number)] = "ocr_layer" if keep else "ocr"
    else:
        source_of[(r.report_id, r.page_number)] = "ocr"
pages_df["text_source"] = [source_of[(r.report_id, r.page_number)] for r in pages_df.itertuples()]

# the lines that are actually used as page text: RapidOCR on image_only pages, the kept layer on ocr_layer pages
scan_lines_df["used"] = (((scan_lines_df["source"] == "rapidocr") & (scan_lines_df["text_class"] == "image_only"))
                         | ((scan_lines_df["source"] == "layer") & (scan_lines_df["text_class"] == "ocr_layer")))

# OCR noise: a weak reading (< 0.80 after the retry) of at most two characters is a mark, not text
# (reviewed on the renders: a misread of the letterhead seal, a pen stroke of a signature)
OCR_NOISE_MAX_CONF, OCR_NOISE_MAX_CHARS = 0.80, 2
scan_lines_df["ocr_noise"] = (scan_lines_df["conf"].fillna(1.0) < OCR_NOISE_MAX_CONF) \
    & (scan_lines_df["text"].str.replace(" ", "").str.len() <= OCR_NOISE_MAX_CHARS) & ~scan_lines_df["figure_internal"]

used = scan_lines_df[scan_lines_df["used"]]
weak = used[(used["conf"] < OCR_NOISE_MAX_CONF) & ~used["figure_internal"]]
print("Text source per page:", pages_df["text_source"].value_counts().to_dict())
print(f"Used scanned-page lines: {len(used)} | figure-internal {int(used['figure_internal'].sum())} "
      f"| OCR noise {int(used['ocr_noise'].sum())} | weak body lines (< {OCR_NOISE_MAX_CONF}) {len(weak)}")
with pd.option_context("display.max_rows", None, "display.max_colwidth", 90):
    display(weak[["report_id", "page_number", "line_no", "conf", "ocr_noise", "text"]])
    print("Figure-internal lines of >= 6 words (reviewed on the renders):")
    display(used[used["figure_internal"] & used["text"].str.split().str.len().ge(6)][["report_id", "page_number", "text"]])

Text source per page: {'native': 446, 'ocr': 115, 'ocr_layer': 19}
Used scanned-page lines: 4778 | figure-internal 417 | OCR noise 2 | weak body lines (< 0.8) 2


,report_id,page_number,line_no,conf,ocr_noise,text
465,H20_BELOW,6,0,0.2644,True,3
912,H20_BELOW,16,27,0.7101,True,V


Figure-internal lines of >= 6 words (reviewed on the renders):


,report_id,page_number,text
697,H20_BELOW,12,Date / Time Dive Computer Computer Model Dive Type Location
2594,MERRICK,16,CT Department of Agriculture Bureau of Aquaculture
3883,QUEEN_OF_THE_WEST,13,railing: 54 inches and 10 feet
4952,UNA_MAS,10,MAXIMUM ENGINE WEIGHT (TWIN) 515 LBS
4953,UNA_MAS,10,THIS BOAT COMPLIES WITH US COAST GUARD SAFETY
4954,UNA_MAS,10,STANDARD IN EFFECT ON DATE OF CERTIFICATION
4955,UNA_MAS,10,"GLACIER BAY, INC., MONROE. WA (360) 794-0444"
4957,UNA_MAS,10,This bout was manufoctured with pride by crafsmen m
4958,UNA_MAS,10,"Mouroe, Washington. U.S.A. The desgn is covered"
4959,UNA_MAS,10,the folowing U S patents. 5.188.049 and DES33( 362


In [21]:
# =========================================================
# 1b.8 Phase 1b validation
# =========================================================
# ---- cache: one entry per OCR'd page, current key, every line with a confidence ----
assert len(ocr_results) == len(OCR_TARGETS) == 134
for (rid, p), res in ocr_results.items():
    assert res["key"] == ocr_cache_key(rid, p) and ocr_cache_path(rid, p).is_file(), (rid, p)
    assert all(0.0 <= l["conf"] <= 1.0 for l in res["lines"]), (rid, p)
assert sorted(f.name for f in OCR_CACHE_DIR.glob("*.json")) == sorted(ocr_cache_path(r, p).name for r, p in ocr_results)
# ---- the retry of weak lines works (evidence of 1b.0: UNA_MAS p.5) ----
una5 = [l for l in ocr_results[("UNA_MAS", 5)]["lines"] if l["retry"] and l["retry"]["kept"] == "retry"]
assert any(l["text"].startswith("vessel capsized, turning over port side first") for l in una5)

# ---- gate: every page used as text passes; every OCR layer is kept ----
assert (gate_df["gate"] == "pass").all(), gate_df[gate_df["gate"] == "fail"]
assert (layer_check_df["decision"] == "keep_layer").all() and len(layer_check_df) == 19
assert pages_df["text_source"].value_counts().to_dict() == {"native": 446, "ocr": 115, "ocr_layer": 19}
every_scanned_page_has_text = used.groupby(["report_id", "page_number"]).size()
assert len(every_scanned_page_has_text) == 134, "Every image_only / ocr_layer page carries text (no bare photo page)."

# ---- reviewed snapshot (2026-09-26) ----
REVIEWED_MIN_DICT_PAGE = ("H20_BELOW", 15, "rapidocr")          # 0.781: readable; diving terms absent from native pages
lowest = gate_df.sort_values("dict_share").iloc[0]
assert (lowest["report_id"], lowest["page_number"], lowest["source"]) == REVIEWED_MIN_DICT_PAGE
REVIEWED_OCR_NOISE = {("H20_BELOW", 6, "3"), ("H20_BELOW", 16, "V")}   # letterhead-seal misread; signature pen stroke
assert set(map(tuple, used.loc[used["ocr_noise"], ["report_id", "page_number", "text"]].values)) == REVIEWED_OCR_NOISE
assert weak["ocr_noise"].all(), "Every weak body line is reviewed OCR noise."
REVIEWED_LONG_FIGURE_LINES = {        # text printed inside figures (checked on the renders, 2026-09-26)
    ("H20_BELOW", 12): 1,             # dive-computer screen header
    ("MERRICK", 16): 1,               # map title 'CT Department of Agriculture Bureau of Aquaculture' (layer)
    ("QUEEN_OF_THE_WEST", 13): 1,     # label on the damage diagram
    ("UNA_MAS", 10): 8,               # capacity plate in the photo
}
long_fig = used[used["figure_internal"] & used["text"].str.split().str.len().ge(6)].groupby(["report_id", "page_number"]).size()
assert long_fig.to_dict() == REVIEWED_LONG_FIGURE_LINES, long_fig.to_dict()
REVIEWED_FIGURE_INTERNAL = 417
assert int(used["figure_internal"].sum()) == REVIEWED_FIGURE_INTERNAL, int(used["figure_internal"].sum())


print("✅ Phase 1b validation passed.")
print(f"OCR'd pages: {len(ocr_results)} (cache) | text source: {pages_df['text_source'].value_counts().to_dict()}")
print(f"Gate: all {len(gate_df)} page readings pass | lowest dict share {lowest['dict_share']} ({REVIEWED_MIN_DICT_PAGE[0]} p.{REVIEWED_MIN_DICT_PAGE[1]})")
print(f"Scanned lines used: {len(used)} | figure-internal {int(used['figure_internal'].sum())} | OCR noise {int(used['ocr_noise'].sum())}")

✅ Phase 1b validation passed.
OCR'd pages: 134 (cache) | text source: {'native': 446, 'ocr': 115, 'ocr_layer': 19}
Gate: all 153 page readings pass | lowest dict share 0.781 (H20_BELOW p.15)
Scanned lines used: 4778 | figure-internal 417 | OCR noise 2


# Phase 2 — Line and row tables + ROI template spec (learned from the sample)

**One line table for the whole corpus (2.1).** Every text line of every page, whatever its source, in one shape:
native lines come from `page_dict()` (superscript footnote markers split off, as in the IMO notebook), scanned pages
contribute the lines chosen in Phase 1b (RapidOCR on `image_only` pages, the kept layer on `ocr_layer` pages) with their
confidence, `figure_internal` and `ocr_noise` flags. `provenance` ∈ {`native`, `ocr_layer`, `ocr`} stays on every line
so OCR text can always be told apart (plan §4.2).

**Rows by baseline (2.2).** One printed row = the lines of one page whose baselines agree (native: ±2.5 pt; OCR: box
bottoms within max(3 pt, 35 % of the box height)). A row's text is its segments in x order, so a vessel-table label
and its value, or "8." and "Recommendations" (BIG_STICK), become one row.

**The ROI template (2.3)** — written from the seeded sample (VICAR_602, CUMBERLAND_II, BIG_STICK, KIM_LY) and the OCR
sample (UNA_MAS), then widened only with the variants the fingerprint (Phase 3) found in the other reports:

| Part | What the template expects | Variants found |
|---|---|---|
| Cover (p.1) | "Report of (the) Investigation into the …", MISLE Activity Number | image-only on 7 reports (OCR'd) |
| Letter start | file block `16732…` + date at the right, letterhead above it | continuation pages repeat file block + date |
| Casualty title | bold capitals, centred, above each letter heading | wording differs between letters of one report |
| Letters | ACTION BY THE COMMANDANT (+ ACTION ON RECOMMENDATION[S]); ENDORSEMENT BY THE COMMANDER, <district> (+ ENDORSEMENT ON RECOMMENDATIONS); ENDORSEMENT BY THE OFFICER IN CHARGE, MARINE INSPECTION (+ COMMENTS ON THE REPORT); EXECUTIVE SUMMARY | "COMMANDANT'S ACTION ON REPORT OF INVESTIGATION"; "ENDORSEMENT/RECOMMENDATIONS"; "MARINE INSPECTIONS"; district name on its own row; "ENDORSEMENT BY THE DISTRICT COMMANDER" |
| Report | INVESTIGATING OFFICER'S REPORT, sections 1–8 | "DISTRICT FORMAL INVESTIGATION REPORT" (SPIRIT_OF_BOSTON); a 2013 **memorandum** (QUEEN_OF_THE_WEST: From/To/Subj/Ref, "Findings of Fact:" …) |
| Section titles | 1 Preliminary Statement · 2 Vessel(s) Involved in the Incident · 3 Deceased, Missing, and/or Injured Persons · 4 Findings of Fact · 5 Analysis · 6 Conclusions · 7 Actions Taken Since the Incident · 8 Recommendations | "Vessels and Motor Vehicle Involved", "Craft Involved", "Record of Deceased, Missing, and Injured", "Finding of Fact"; memo "Vessel Data:", "Personnel Data:"; numbers missing (CUMBERLAND_II 7–8) or slipped (BIG_STICK "5. Conclusions", LA6893ER "4. Analysis") |
| Numbering | `4.1.1.` paragraphs, depth = number of levels | trailing dot optional ("6.4 Evidence …") |
| Vessel block | label column + value column: Official Name, Identification Number, Flag, Vessel Class/Type/Sub-Type, Build Year, Gross Tonnage, Length, Beam/Width, Draft/Depth, Main/Primary Propulsion, Owner, Operator | "Vessel Name:", "Gross Tons:", memo "Name:", "Official Number:", "Service:" |
| Persons table | Relationship to Vessel · Sex · Age · Status | "Relationship to Motor Vehicle"; memo "Deceased Age Position Period of Employment" |
| Figures | caption "Figure N." / "Figure N:" / "Figure N -" under the image | photo credits run on the next caption row |
| Layout | page number at the bottom centre (arabic or roman), "Enclosure (1)" footer, running header on PROWLER_ATTESSA_IV / QUEEN_OF_THE_WEST, table of contents (CAROL_ANN, CONCHRD, PROWLER_ATTESSA_IV, SPIRIT_OF_BOSTON) | |
| Signature block | NAME in capitals, rank ", U.S. Coast Guard", office lines ("By Direction", "Chief, …") | the signature itself is usually redacted (black box) |

In [22]:
# =========================================================
# 2.1 Unified line table (native + OCR, all 23 reports)
# =========================================================
BODY_SIZE = 12.0                       # Word body text (Times New Roman 12 pt) in the native reports
MIN_IMAGE_W, MIN_IMAGE_H = 60, 40      # an image placement large enough to carry labels


def is_bold_span(s: dict) -> bool:
    return bool(s["flags"] & 16) or "Bold" in s["font"]


# ---- character repairs (reviewed, 2026-09-26) ----
# (1) Native text: control characters 0x03–0x1F are glyph codes of subset fonts without a Unicode map, shifted by 29
#     (0x03 -> space, 0x11 -> '.', 0x13–0x1C -> '0'–'9'). Evidence: CUMBERLAND_II '4.1.1\x11 At 1151' (paragraph dots)
#     and '\x1a\x11 Actions Taken…' / '\x1b\x11 Recommendations' between 6.7 and 8.1 (-> '7.', '8.'); BIG_STICK file date
#     '2\x19 Jan 2026' (-> 26 Jan); NC3250BW '2\x17 October 2025' (-> 24 October); LA6893ER '\x03' spaces and page numbers.
# (2) OCR / OCR-layer text: control characters are misreadings of the letterhead seal -> removed.
# (3) CUYAHOGA's font maps the right quote to '¶' and the en dash to '±' ("Officer¶s Report", "Department¶s swift
#     actions", "Fire Department ± Fire Prevention Bureau") -> repaired for that report only ('½' is a real fraction).
GLYPH_SHIFT = 29
CONTROL_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")
REPORT_CHAR_REPAIRS = {"CUYAHOGA": {"¶": "’", "±": "–"}}
text_repairs = Counter()


def clean_text(t: str, rid: str, provenance: str) -> str:
    if CONTROL_RE.search(t):
        if provenance == "native":
            text_repairs[(rid, "glyph_shift_29")] += len(CONTROL_RE.findall(t))
            t = CONTROL_RE.sub(lambda m: chr(ord(m.group(0)) + GLYPH_SHIFT), t)
        else:
            text_repairs[(rid, "ocr_control_removed")] += len(CONTROL_RE.findall(t))
            t = CONTROL_RE.sub("", t)
    for bad, good in REPORT_CHAR_REPAIRS.get(rid, {}).items():
        if bad in t:
            text_repairs[(rid, f"{bad}->{good}")] += t.count(bad)
            t = t.replace(bad, good)
    return " ".join(unicodedata.normalize("NFC", t).replace("­", "").split())


line_records = []
for r in pages_df.itertuples():
    rid, p = r.report_id, r.page_number
    if r.text_source == "native":
        page = docs[rid][p - 1]
        images = [i["bbox"] for i in page.get_image_info()
                  if i["bbox"][2] - i["bbox"][0] >= MIN_IMAGE_W and i["bbox"][3] - i["bbox"][1] >= MIN_IMAGE_H]
        k = 0
        for b in page_dict(rid, p)["blocks"]:
            for l in b["lines"]:
                spans = [s for s in l["spans"] if s["text"].strip()]
                if not spans:
                    continue
                main = max(spans, key=lambda s: len(s["text"].strip()))
                sup = lambda s: bool(s["flags"] & 1) and s["size"] < 0.8 * main["size"]
                text = clean_text("".join(s["text"] for s in l["spans"] if not sup(s)), rid, "native")
                if not text:
                    continue
                x0, y0, x1, y1 = l["bbox"]
                cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
                line_records.append({
                    "line_key": f"{rid}|{p:03d}|n{k:04d}", "report_id": rid, "page_number": p, "provenance": "native",
                    "x0": round(x0, 1), "y0": round(y0, 1), "x1": round(x1, 1), "y1": round(y1, 1),
                    "baseline": round(main["origin"][1], 1), "text": text,
                    "markers": [s["text"].strip() for s in l["spans"] if sup(s) and s["text"].strip()],
                    "size": round(main["size"], 1), "bold": all(is_bold_span(s) for s in spans if not sup(s)),
                    "conf": np.nan, "in_image": any(a[0] <= cx <= a[2] and a[1] <= cy <= a[3] for a in images),
                    "figure_internal": False, "ocr_noise": False})
                k += 1
    else:
        used_here = scan_lines_df[(scan_lines_df["report_id"] == rid) & (scan_lines_df["page_number"] == p)
                                  & scan_lines_df["used"]]
        for s in used_here.itertuples():
            x0, y0, x1, y1 = s.bbox
            text = clean_text(s.text, rid, "ocr" if r.text_source == "ocr" else "ocr_layer")
            if not text:
                continue
            line_records.append({
                "line_key": f"{rid}|{p:03d}|{'o' if s.source == 'rapidocr' else 'l'}{s.line_no:04d}",
                "report_id": rid, "page_number": p, "provenance": "ocr" if r.text_source == "ocr" else "ocr_layer",
                "x0": x0, "y0": y0, "x1": x1, "y1": y1, "baseline": round(y1 - 0.2 * (y1 - y0), 1), "text": text,
                "markers": [], "size": np.nan, "bold": None, "conf": s.conf, "in_image": False,
                "figure_internal": bool(s.figure_internal), "ocr_noise": bool(s.ocr_noise)})

lines_df = pd.DataFrame(line_records)
lines_df["n_chars"] = lines_df["text"].str.len()
print("Lines:", len(lines_df), lines_df["provenance"].value_counts().to_dict())
print("Lines per report:", lines_df.groupby("report_id").size().to_dict())
print("Character repairs:", dict(text_repairs))
print("Native lines inside an image placement:", int(lines_df["in_image"].sum()),
      "| OCR figure-internal:", int(lines_df["figure_internal"].sum()), "| OCR noise:", int(lines_df["ocr_noise"].sum()))

Lines: 18999 {'native': 14224, 'ocr': 4085, 'ocr_layer': 690}
Lines per report: {'BIG_STICK': 505, 'CAROL_ANN': 1748, 'CONCHRD': 1456, 'CUMBERLAND_II': 438, 'CUYAHOGA': 1169, 'GENERAL_MACARTHUR': 442, 'GOOD_GULF': 353, 'H20_BELOW': 574, 'KIM_LY': 496, 'LA6893ER': 681, 'LADY_ELLEN': 739, 'LITTLE_G': 415, 'MERRICK': 640, 'MICHIPICOTEN': 683, 'NC3250BW': 352, 'PROWLER_ATTESSA_IV': 702, 'QUEEN_OF_THE_WEST': 976, 'SPIRIT_OF_BOSTON': 3307, 'TOO_SHORT': 485, 'UNA_MAS': 586, 'VICAR_602': 431, 'WAYMON_BOYD': 1196, 'WILDERNESS_I': 625}
Character repairs: {('BIG_STICK', 'glyph_shift_29'): 1, ('CONCHRD', 'ocr_control_removed'): 1, ('CUMBERLAND_II', 'glyph_shift_29'): 18, ('CUYAHOGA', '±->–'): 2, ('CUYAHOGA', '¶->’'): 6, ('LA6893ER', 'glyph_shift_29'): 20, ('MERRICK', 'ocr_control_removed'): 1, ('NC3250BW', 'glyph_shift_29'): 1, ('TOO_SHORT', 'ocr_control_removed'): 1, ('VICAR_602', 'ocr_control_removed'): 3}
Native lines inside an image placement: 63 | OCR figure-internal: 417 | OCR noise: 2


In [23]:
# =========================================================
# 2.2 Rows by baseline
# =========================================================
NATIVE_ROW_TOL = 2.5


def row_tolerance(r) -> float:
    return NATIVE_ROW_TOL if r.provenance == "native" else max(3.0, 0.35 * (r.y1 - r.y0))


row_records, row_of_line = [], {}
for (rid, p), grp in lines_df.groupby(["report_id", "page_number"], sort=True):
    grp = grp.sort_values(["baseline", "x0"])
    members, base = [], None
    groups = []
    for r in grp.itertuples():
        if members and abs(r.baseline - base) > row_tolerance(r):
            groups.append(members)
            members = []
        if not members:
            base = r.baseline
        members.append(r.Index)
    if members:
        groups.append(members)
    # a baseline group is split into rows whose segments never overlap horizontally and (native) share a font size:
    # a caption printed across a body line (SPIRIT_OF_BOSTON p.18), two captions on one baseline (WAYMON_BOYD p.17)
    # or a stray glyph of an OCR layer (MERRICK p.15) stays a row of its own
    split_groups = []
    for m in groups:
        subrows: list[list[int]] = []
        for i in sorted(m, key=lambda i: lines_df.at[i, "x0"]):
            li = lines_df.loc[i]
            for sr in subrows:
                last = lines_df.loc[sr[-1]]
                same_size = li["provenance"] != "native" or abs(li["size"] - last["size"]) <= 1.5
                if li["x0"] >= last["x1"] - 2.0 and same_size:
                    sr.append(i)
                    break
            else:
                subrows.append([i])
        split_groups.extend(subrows)
    groups = sorted(split_groups, key=lambda m: (lines_df.loc[m, "y0"].min(), lines_df.loc[m, "x0"].min()))
    for k, m in enumerate(groups):
        seg = lines_df.loc[m].sort_values("x0")
        row_id = f"{rid}|{p:03d}|r{k:03d}"
        native = seg["provenance"].eq("native").all()
        row_records.append({
            "row_id": row_id, "report_id": rid, "page_number": p,
            "provenance": seg["provenance"].iloc[0],
            "x0": float(seg["x0"].min()), "y0": float(seg["y0"].min()), "x1": float(seg["x1"].max()),
            "y1": float(seg["y1"].max()), "baseline": float(seg["baseline"].min()),
            "text": " ".join(seg["text"]), "segments": [(float(a), float(b), t) for a, b, t in zip(seg["x0"], seg["x1"], seg["text"])],
            "line_keys": seg["line_key"].tolist(), "markers": [mk for ms in seg["markers"] for mk in ms],
            "bold": bool(seg["bold"].all()) if native else None,
            "size": float(seg["size"].max()) if native else np.nan,
            "height": float((seg["y1"] - seg["y0"]).median()),
            "conf": float(seg["conf"].min()) if not native else np.nan,
            "in_image": bool(seg["in_image"].any()),
            "figure_internal": bool(seg["figure_internal"].all()), "ocr_noise": bool(seg["ocr_noise"].all()),
        })
        for i in m:
            row_of_line[i] = row_id

rows_df = pd.DataFrame(row_records).sort_values(["report_id", "page_number", "y0", "x0"]).reset_index(drop=True)
lines_df["row_id"] = lines_df.index.map(row_of_line)
rows_df["centre"] = (rows_df["x0"] + rows_df["x1"]) / 2
print(f"Rows: {len(rows_df)} from {len(lines_df)} lines | rows with several segments: "
      f"{int(rows_df['segments'].map(len).gt(1).sum())}")
print(rows_df.loc[(rows_df["report_id"] == "BIG_STICK") & rows_df["text"].str.contains("Recommendations$"),
                  ["page_number", "text", "segments"]].to_string())

Rows: 17471 from 18999 lines | rows with several segments: 1178
     page_number                text                                            segments
465           18  8. Recommendations  [(72.0, 84.3, 8.), (94.6, 192.2, Recommendations)]


In [24]:
# =========================================================
# 2.3 ROI template spec (reviewed, 2026-09-26)
# =========================================================
ROI_TEMPLATE = {
    # letter headings: centred capitals; a heading may continue on the next row (district name)
    "letters": {
        "action_commandant": r"^(ACTION BY THE COMMANDANT|COMMANDANT.?S ACTION ON (THE )?REPORT OF (THE )?INVESTIGATION)$",
        "action_on_recommendations": r"^ACTION ON (THE )?RECOMMENDATIONS?$",
        "endorsement_district": r"^ENDORSEMENT BY THE (DISTRICT )?COMMANDER\b.*$",
        "endorsement_ocmi": r"^ENDORSEMENT BY THE OFFICER IN CHARGE,? MARINE INSPECTIONS?$",
        "endorsement_on_recommendations": r"^(ENDORSEMENTS? ON (THE )?RECOMMENDATIONS?|ENDORSEMENT/RECOMMENDATIONS"
                                          r"|ENDORSEMENT/ACTION ON (SAFETY |ADMINISTRATIVE )?RECOMMENDATIONS)$",
        "comments_on_report": r"^COMMENTS ON THE\s?REPORT$",
        "executive_summary": r"^EXECUTIVE SUMMARY:?$",
        "io_report": r"^(INVESTIGATING\s?OFFICER.?S REPORT|DISTRICT FORMAL INVESTIGATION REPORT|MEMORANDUM)$",
    },
    "front_matter": {
        "table_of_contents": r"^(TABLE OF CONTENTS|Table of Contents)$",
        "list_of_acronyms": r"^LIST OF ACRONYMS$",
    },
    # report sections: number optional (missing or slipped in some reports), trailing colon optional (memorandum)
    "sections": {
        1: r"Preliminary Statement",
        2: r"(Vessels?|Vessel\(s\)|Craft)( and Motor Vehicles?)? Involved( in the (Incident|Casualty|Accident))?|Vessel Data"
           r"|The [Vv]essel",
        3: r"(Record of )?Deceased,? Missing,? and(/or)? Injured( Persons)?|Personnel Data",
        4: r"Findings? of Fact",
        5: r"Analysis",
        6: r"Conclusions?",
        7: r"Actions Taken Since the (Incident|Casualty|Accident)",
        8: r"Recommendations?",
    },
    "section_suffix": r"(\s*[-–—]\s*.+)?",      # SPIRIT_OF_BOSTON: "3. Findings of Fact – The Incident", "4. … - Additional…"
    "paragraph_number": r"^(\d{1,2}(?:\.\d{1,2}){1,5})\.?(?=\s|$)",
    "file_block": r"^16732\b",
    "cover": r"(?:REPORT OF (?:THE )?(?:DISTRICT FORMAL )?INVESTIGATION|INCIDENT INVESTIGATION REPORT)",
    # figure captions, table captions (SPIRIT_OF_BOSTON) and "Above : Photograph of …" (PROWLER_ATTESSA_IV)
    "caption": r"^(?:Fig(?:ure|\.)?\s*\d+\s*[.:\-–—]?|Table\s*\d+\s*[.:\-–—]|Above\s*:\s*Photo)",
    "page_number": r"^([ivxlc]{1,5}|\d{1,3})$",
    "enclosure_footer": r"^Enclosure\s*\(\d+\)$",
    "vessel_label": r"^(Official Name/State Number|Official Name|Vessel Name|Name|Identification Number|Vessel Identification Number|Official Number"
                    r"|Flag|Vessel Class/Type/Sub-?Type|Service|Build Year|Gross Tonnage|Gross Tons|Length|Beam/Width"
                    r"|Draft/Depth|Draft|Hull Material|Main/Primary Propulsion|Main Primary Propulsion|Propulsion|Owner"
                    r"|Operator)\s*:?",
    "persons_header": r"^(?:Relationship to (?:Vessel|Motor Vehicle|Craft)\b.*|Deceased Age Position.*|Name \(First.*Status)",
    "memo_header": r"^(MEMORANDUM|From:|To:|Subj:|Ref:)",
    "rank": r"(Captain|Commander|Lieutenant( Commander)?|Rear Admiral|Vice Admiral|Chief Warrant Officer|Ensign)\b.*"
            r"U\.\s?S\.\s?Coast Guard",
}
LETTER_RE = {k: re.compile(v) for k, v in ROI_TEMPLATE["letters"].items()}
FRONT_RE = {k: re.compile(v) for k, v in ROI_TEMPLATE["front_matter"].items()}
SECTION_RE = {k: re.compile(rf"^(?:(?P<num>\d{{1,2}})\s*\.?\s*)?(?P<title>{v}){ROI_TEMPLATE['section_suffix']}\s*:?$",
                            re.IGNORECASE) for k, v in ROI_TEMPLATE["sections"].items()}
PARA_NUM_RE = re.compile(ROI_TEMPLATE["paragraph_number"])
FILE_BLOCK_RE = re.compile(ROI_TEMPLATE["file_block"])
CAPTION_START_RE = re.compile(ROI_TEMPLATE["caption"], re.IGNORECASE)
PAGE_NUMBER_RE = re.compile(ROI_TEMPLATE["page_number"], re.IGNORECASE)
ENCLOSURE_FOOTER_RE = re.compile(ROI_TEMPLATE["enclosure_footer"])
VESSEL_LABEL_RE = re.compile(ROI_TEMPLATE["vessel_label"], re.IGNORECASE)
PERSONS_HEADER_RE = re.compile(ROI_TEMPLATE["persons_header"], re.IGNORECASE)
MEMO_HEADER_RE = re.compile(ROI_TEMPLATE["memo_header"])
RANK_RE = re.compile(ROI_TEMPLATE["rank"])


def letter_of(text: str) -> str | None:
    t = " ".join(text.split()).upper().replace("’", "'")
    return next((k for k, rx in LETTER_RE.items() if rx.match(t)), None)


def section_of(text: str) -> tuple[int, str | None] | None:
    t = " ".join(text.split())
    for k, rx in SECTION_RE.items():
        if m := rx.match(t):
            return k, m.group("num")
    return None


assert letter_of("ACTION BY THE COMMANDANT") == "action_commandant"
assert letter_of("INVESTIGATING OFFICER’S REPORT") == letter_of("INVESTIGATINGOFFICER'S REPORT") == "io_report"
assert section_of("2. Vessels and Motor Vehicle Involved in the Incident") == (2, "2")
assert section_of("Actions Taken Since the Incident") == (7, None) and section_of("5. Conclusions") == (6, "5")
assert section_of("Findings of Fact:") == (4, None) and section_of("8.1. Safety Recommendation:") is None
print("ROI template ready:", len(LETTER_RE), "letter headings,", len(SECTION_RE), "sections")

ROI template ready: 8 letter headings, 8 sections


In [25]:
# =========================================================
# 2.4 Phase 2 validation
# =========================================================
assert lines_df["line_key"].is_unique and rows_df["row_id"].is_unique
assert lines_df["row_id"].notna().all(), "Every line belongs to exactly one row."
assert sorted(k for ks in rows_df["line_keys"] for k in ks) == sorted(lines_df["line_key"])
assert set(lines_df.groupby(["report_id", "page_number"]).groups) == set(zip(pages_df["report_id"], pages_df["page_number"])), \
    "Every page contributes text (all 134 scanned pages carry text, Phase 1b)."
assert (lines_df.loc[lines_df["provenance"] == "ocr", "conf"].between(0, 1)).all()
assert lines_df.loc[lines_df["provenance"] != "ocr", "ocr_noise"].eq(False).all()
# rows never join two lines that overlap horizontally (that would be two printed rows)
for r in rows_df[rows_df["segments"].map(len) > 1].itertuples():
    xs = sorted(lines_df.set_index("line_key").loc[r.line_keys, ["x0", "x1"]].itertuples(index=False))
    assert all(a.x1 <= b.x0 + 2.0 for a, b in zip(xs, xs[1:])), f"Overlapping segments in one row: {r.row_id} {r.text[:60]!r}"

# ---- reviewed snapshot (2026-09-26) ----
REVIEWED_LINES = {"native": 14224, "ocr": 4085, "ocr_layer": 690}   # 14 lines were control characters only (no text)
assert lines_df["provenance"].value_counts().to_dict() == REVIEWED_LINES, lines_df["provenance"].value_counts().to_dict()
print("✅ Phase 2 validation passed.")
print("Lines:", len(lines_df), REVIEWED_LINES, "| rows:", len(rows_df))

✅ Phase 2 validation passed.
Lines: 18999 {'native': 14224, 'ocr': 4085, 'ocr_layer': 690} | rows: 17471


# Phase 3 — Template fingerprint of all 23 reports (text only) + 3b individual inspection of the deviating ones

Every report is compared with the ROI template of Phase 2 using the row texts only (no layout parsing): cover page,
letter headings, the start of the investigating officer's report, the eight sections (found, order, printed number),
vessel-block labels, persons table, figure captions, table of contents, running header. What differs from the
template is a **deviation**; a report without deviation is `conformant`.

**3b** — each deviating report was then inspected individually (only those), and every deviation receives a recorded
decision: how the parser handles it. The reviewed list is asserted against the computed one, so a new or vanished
deviation stops the notebook.

In [26]:
# =========================================================
# 3.1 Fingerprint
# =========================================================
TOP_BAND, BOTTOM_BAND = 90.0, 700.0
COVER_RE = re.compile(ROI_TEMPLATE["cover"], re.IGNORECASE)


def mask_digits(t: str) -> str:
    return re.sub(r"\d+", "#", " ".join(t.split()).upper())


fingerprint_records, section_hits = [], []
for rid, g in rows_df.groupby("report_id", sort=True):
    g = g.sort_values(["page_number", "y0"])
    letters = [(r.page_number, r.y0, letter_of(r.text)) for r in g.itertuples() if letter_of(r.text)]
    front = {k: sorted({r.page_number for r in g.itertuples() if rx.match(" ".join(r.text.split()))})
             for k, rx in FRONT_RE.items()}
    toc_pages = set(front["table_of_contents"])
    io = [(p, y) for p, y, k in letters if k == "io_report"]
    # the report body starts at its heading; two reports print none (CUYAHOGA, LITTLE_G) and start directly with
    # "1. Preliminary Statement"; the convening MEMORANDUM of SPIRIT_OF_BOSTON is front matter (its report has a heading)
    real_io = [(p, y) for (p, y), k in zip([(a, b) for a, b, _ in letters], [k for _, _, k in letters])
               if k == "io_report" and not g.loc[(g["page_number"] == p) & (g["y0"] == y), "text"].str.strip().eq("MEMORANDUM").all()]
    prelim = [(r.page_number, r.y0) for r in g.itertuples()
              if r.x0 < 150 and section_of(r.text) and section_of(r.text)[0] == 1 and r.page_number not in toc_pages]
    io_start = (real_io or io or prelim or [(int(g["page_number"].max()) + 1, 0.0)])[0]
    secs = []
    for r in g.itertuples():
        if (r.page_number, r.y0) < io_start or r.page_number in toc_pages or r.x0 > 150 or len(r.text) > 90:
            continue
        hit = section_of(r.text)
        if hit and hit[0] not in [s[0] for s in secs]:
            secs.append((hit[0], hit[1], r.page_number, r.row_id))
    for s in secs:
        section_hits.append({"report_id": rid, "section": s[0], "printed_number": s[1], "page_number": s[2], "row_id": s[3]})
    # running header: a top-band row repeated on >= 3 pages of the report body (letterheads repeat on letter pages only)
    top = g[(g["y1"] <= TOP_BAND) & (g["page_number"] > io_start[0])]
    repeated = top.groupby(top["text"].map(mask_digits))["page_number"].nunique()
    fingerprint_records.append({
        "report_id": rid,
        "cover": bool(g[g["page_number"] == 1]["text"].str.contains(COVER_RE).any()),
        "letters": [k for _, _, k in letters],
        "memorandum": "MEMORANDUM" in set(g["text"].str.strip()) and not real_io,
        "io_start_page": io_start[0] if (io or prelim) else None,
        "report_heading": bool(io),
        "sections": [s[0] for s in secs], "printed": [s[1] for s in secs],
        "vessel_blocks": int(g["text"].str.match(r"^(?:Official Name|Vessel Name|Name)(?:/State Number)?\s*:", case=False).sum()),
        "persons_header": int(g["text"].str.match(PERSONS_HEADER_RE).sum()),
        "captions": int(g["text"].str.match(CAPTION_START_RE).sum()),
        "toc_pages": sorted(toc_pages), "acronym_pages": front["list_of_acronyms"],
        "running_header_rows": int((repeated >= 3).sum()),
    })
fingerprint_df = pd.DataFrame(fingerprint_records)
section_hits_df = pd.DataFrame(section_hits)


def deviations_of(f) -> list[str]:
    dev = []
    if not f["cover"]:
        dev.append("no_cover_title")
    if f["memorandum"]:
        dev.append("memorandum_format")
    if not f["report_heading"]:
        dev.append("no_report_heading")
    if f["letters"].count("action_commandant") != 1:
        dev.append(f"commandant_actions_{f['letters'].count('action_commandant')}")
    missing = [k for k in range(1, 9) if k not in f["sections"]]
    if missing:
        dev.append("sections_missing_" + "".join(map(str, missing)))
    if f["sections"] != sorted(f["sections"]):
        dev.append("sections_out_of_order")
    for k, n in zip(f["sections"], f["printed"]):
        if n is None and not f["memorandum"]:
            dev.append(f"section_{k}_unnumbered")
        elif n is not None and int(n) != k:
            dev.append(f"section_{k}_printed_{n}")
    if f["vessel_blocks"] == 0:
        dev.append("no_vessel_block_label")
    if f["persons_header"] == 0:
        dev.append("no_persons_table")
    if f["toc_pages"]:
        dev.append("table_of_contents")
    if f["acronym_pages"]:
        dev.append("list_of_acronyms")
    if f["running_header_rows"]:
        dev.append("running_header")
    return dev


fingerprint_df["deviations"] = fingerprint_df.apply(deviations_of, axis=1)
fingerprint_df["status"] = np.where(fingerprint_df["deviations"].map(len) == 0, "conformant", "deviating")
with pd.option_context("display.max_rows", None, "display.max_colwidth", 120, "display.width", 250):
    display(fingerprint_df[["report_id", "sections", "printed", "vessel_blocks", "persons_header", "captions",
                            "running_header_rows", "status", "deviations"]])
print(fingerprint_df["status"].value_counts().to_dict())

,report_id,sections,printed,vessel_blocks,persons_header,captions,running_header_rows,status,deviations
0,BIG_STICK,"[1, 2, 3, 4, 5, 6, 7, 8]","[1, 2, 3, 4, 5, 5, 7, 8]",1,1,7,0,deviating,[section_6_printed_5]
1,CAROL_ANN,"[1, 2, 3, 4, 5, 6, 7, 8]","[1, 2, 3, 4, 5, 6, 7, 8]",1,1,13,0,deviating,[table_of_contents]
2,CONCHRD,"[1, 2, 3, 4, 5, 6, 7, 8]","[1, 2, 3, 4, 5, 6, 7, 8]",1,1,17,0,deviating,"[table_of_contents, list_of_acronyms]"
3,CUMBERLAND_II,"[1, 2, 3, 4, 5, 6, 7, 8]","[1, 2, 3, 4, 5, 6, 7, 8]",2,1,4,0,conformant,[]
4,CUYAHOGA,"[1, 2, 3, 4, 5, 6, 7, 8]","[1, 2, 3, 4, 5, 6, 7, 8]",1,1,16,0,deviating,[no_report_heading]
5,GENERAL_MACARTHUR,"[1, 2, 3, 4, 5, 6, 7, 8]","[1, 2, 3, 4, 5, 6, 7, 8]",1,1,14,0,conformant,[]
6,GOOD_GULF,"[1, 2, 3, 4, 5, 6, 7, 8]","[1, 2, 3, 4, 5, 6, 7, 8]",1,1,1,0,conformant,[]
7,H20_BELOW,"[1, 2, 3, 4, 5, 6, 7, 8]","[1, 2, 3, 4, 5, 6, 7, 8]",1,1,4,0,conformant,[]
8,KIM_LY,"[1, 2, 3, 4, 5, 6, 7, 8]","[1, 2, 3, 4, 5, 6, 7, 8]",1,1,3,0,conformant,[]
9,LA6893ER,"[1, 2, 3, 4, 5, 6, 7, 8]","[1, 2, 3, 4, 4, 6, 7, 8]",1,1,2,0,deviating,[section_5_printed_4]


{'conformant': 12, 'deviating': 11}


In [27]:
# =========================================================
# 3.2 (3b) Individual inspection of the deviating reports — reviewed decisions (2026-09-26)
# =========================================================
# Every deviation of 3.1 was inspected on the report itself (rows / renders); the decision says how the parser handles
# it. Numbering slips are recorded, never "corrected": the canonical section comes from the printed title.
REVIEWED_DEVIATIONS = {
    "BIG_STICK": {
        "section_6_printed_5": "numbering slip (plan §0): heading printed '5. Conclusions' above paragraphs 6.1–6.7 -> "
                               "canonical section 6 from the title; slip recorded"},
    "CAROL_ANN": {"table_of_contents": "front matter: table of contents (p.17) -> metadata, cross-checked with the headings"},
    "CONCHRD": {
        "table_of_contents": "front matter: table of contents (p.6) -> metadata, cross-checked with the headings",
        "list_of_acronyms": "front matter: list of acronyms (p.7) -> one glossary unit (expands the acronyms of the report)"},
    "CUYAHOGA": {"no_report_heading": "the investigating officer's report prints no heading: it starts at "
                                      "'1. Preliminary Statement' (p.10) -> report start anchored there"},
    "LA6893ER": {"section_5_printed_4": "numbering slip: heading printed '4. Analysis' after '4. Findings of Fact' -> "
                                        "canonical section 5 from the title; slip recorded"},
    "LITTLE_G": {"no_report_heading": "no report heading printed: starts at '1. Preliminary Statement' (p.6) -> anchored there"},
    "MICHIPICOTEN": {"no_persons_table": "Section 3 prints 'None' (no one killed, missing or injured) -> no persons rows"},
    "PROWLER_ATTESSA_IV": {
        "table_of_contents": "front matter: table of contents (p.6) -> metadata, cross-checked with the headings",
        "running_header": "casualty title + file number + date repeated at the top of every report page -> layout"},
    "QUEEN_OF_THE_WEST": {
        "memorandum_format": "2013 memorandum ROI (MEMORANDUM, From/To/Subj/Ref, 'Executive Summary:', 'Vessel Data:', "
                             "'Personnel Data:', 'Findings of Fact:' …, paragraphs numbered 1., 2. …) -> headings by "
                             "title, memorandum header -> metadata, 'Vessel Data' = section 2, 'Personnel Data' = section 3",
        "commandant_actions_2": "two Commandant letters: closure of the investigation (20 Oct 2022) and action on the 17 "
                                "recommendations (31 Aug 2026) -> both kept; version_date = the later action",
        "sections_missing_17": "the memorandum has no Preliminary Statement and no Actions Taken section -> nothing to parse",
        "running_header": "file number, subject line and date repeated on every memorandum page -> layout"},
    "SPIRIT_OF_BOSTON": {
        "sections_missing_3": "no Section 3: no one was killed, missing or injured; the Findings of Fact are printed as two "
                              "numbered sections '3. … – The Incident' and '4. … – Additional/Supporting Information'",
        "section_4_printed_3": "both Findings parts -> canonical section 4 (two parts, printed numbers recorded)",
        "no_persons_table": "no casualties -> no persons rows",
        "table_of_contents": "front matter: table of contents (p.12) -> metadata, cross-checked with the headings"},
    "VICAR_602": {"no_persons_table": "Section 3 is one sentence ('There were no deceased, missing, or injured personnel "
                                      "…') -> kept as a paragraph"},
}

actual_dev = {r.report_id: set(r.deviations) for r in fingerprint_df.itertuples() if r.deviations}
reviewed_dev = {k: set(v) for k, v in REVIEWED_DEVIATIONS.items()}
assert actual_dev == reviewed_dev, {
    "not reviewed": {k: actual_dev[k] - reviewed_dev.get(k, set()) for k in actual_dev if actual_dev[k] - reviewed_dev.get(k, set())},
    "vanished": {k: reviewed_dev[k] - actual_dev.get(k, set()) for k in reviewed_dev if reviewed_dev[k] - actual_dev.get(k, set())}}
deviation_decisions_df = pd.DataFrame([{"report_id": k, "deviation": d, "decision": t}
                                       for k, v in REVIEWED_DEVIATIONS.items() for d, t in v.items()])
with pd.option_context("display.max_colwidth", 140, "display.max_rows", None):
    display(deviation_decisions_df)

,report_id,deviation,decision
0,BIG_STICK,section_6_printed_5,numbering slip (plan §0): heading printed '5. Conclusions' above paragraphs 6.1–6.7 -> canonical section 6 from the title; slip recorded
1,CAROL_ANN,table_of_contents,"front matter: table of contents (p.17) -> metadata, cross-checked with the headings"
2,CONCHRD,table_of_contents,"front matter: table of contents (p.6) -> metadata, cross-checked with the headings"
3,CONCHRD,list_of_acronyms,front matter: list of acronyms (p.7) -> one glossary unit (expands the acronyms of the report)
4,CUYAHOGA,no_report_heading,the investigating officer's report prints no heading: it starts at '1. Preliminary Statement' (p.10) -> report start anchored there
5,LA6893ER,section_5_printed_4,numbering slip: heading printed '4. Analysis' after '4. Findings of Fact' -> canonical section 5 from the title; slip recorded
6,LITTLE_G,no_report_heading,no report heading printed: starts at '1. Preliminary Statement' (p.6) -> anchored there
7,MICHIPICOTEN,no_persons_table,"Section 3 prints 'None' (no one killed, missing or injured) -> no persons rows"
8,PROWLER_ATTESSA_IV,table_of_contents,"front matter: table of contents (p.6) -> metadata, cross-checked with the headings"
9,PROWLER_ATTESSA_IV,running_header,casualty title + file number + date repeated at the top of every report page -> layout


In [28]:
# =========================================================
# 3.3 Phase 3 validation
# =========================================================
assert len(fingerprint_df) == 23 and fingerprint_df["cover"].all(), "Every report has a cover title (p.1)."
assert fingerprint_df["io_start_page"].notna().all(), "Every report has a findable start of the report body."
fp = fingerprint_df.set_index("report_id")
assert fp.loc[["CUMBERLAND_II", "KIM_LY", "UNA_MAS"], "status"].eq("conformant").all()
# the template holds for every report that is not the memorandum: sections 1–8 in order (SPIRIT_OF_BOSTON: no section 3)
for r in fingerprint_df[~fingerprint_df["memorandum"]].itertuples():
    expected = [k for k in range(1, 9) if not (r.report_id == "SPIRIT_OF_BOSTON" and k == 3)]
    assert r.sections == expected, (r.report_id, r.sections)
# ---- reviewed snapshot (2026-09-26) ----
REVIEWED_STATUS = {"conformant": 12, "deviating": 11}
assert fingerprint_df["status"].value_counts().to_dict() == REVIEWED_STATUS, fingerprint_df["status"].value_counts().to_dict()
NUMBERING_SLIPS = {("BIG_STICK", 6): "5", ("LA6893ER", 5): "4", ("SPIRIT_OF_BOSTON", 4): "3"}
slips = {(r.report_id, r.section): r.printed_number for r in section_hits_df.itertuples()
         if pd.notna(r.printed_number) and int(r.printed_number) != r.section}
assert slips == NUMBERING_SLIPS, slips
print("✅ Phase 3 validation passed.")
print("Fingerprint:", fingerprint_df["status"].value_counts().to_dict(), "| deviations reviewed:", len(deviation_decisions_df),
      "| numbering slips:", NUMBERING_SLIPS)

✅ Phase 3 validation passed.
Fingerprint: {'conformant': 12, 'deviating': 11} | deviations reviewed: 19 | numbering slips: {('BIG_STICK', 6): '5', ('LA6893ER', 5): '4', ('SPIRIT_OF_BOSTON', 4): '3'}


# Phase 4 — Template-driven structured extraction

One parser for all 23 reports (plan §1.1): it walks the rows of each report in reading order and gives every row a
**role** from the ROI template (Phase 2) and the reviewed deviations (3b). No row is left without a role.

| Role | Rows | Destination |
|---|---|---|
| `page_number`, `enclosure_footer`, `running_header`, `letterhead` | page furniture | layout (never chunked) |
| `file_block`, `file_date` | "16732/IIA # …" + the date under it | metadata (the Commandant's date = `version_date`) |
| `casualty_title` | the capitalised casualty title above each letter / report heading | metadata (report title) |
| `cover` | page 1 ("Report of the Investigation into …", MISLE Activity Number) | metadata |
| `toc`, `memo_header`, `enclosures` | table of contents, From/To/Subj/Ref, "Enclosures: (1) …" | metadata |
| `signature` | the signer's name, rank and office under a letter | excluded (personal names, no retrieval value; the letter is identified by its heading) |
| `letter_heading`, `section_heading`, `front_heading` | template headings | structure (titles, paths) |
| `caption` | "Figure N …" and its continuation rows | figure units (ADR-020) |
| `footnote` | native notes at the page bottom (smaller type, numbered) | notes policy (Phase 5) |
| `figure_internal`, `ocr_noise` | text inside images / OCR marks (Phase 1b) | excluded with reason |
| `body` | everything else | paragraphs (Phases 6–8) |

Letters are delimited by their headings; the report body starts at its heading or, where none is printed
(CUYAHOGA, LITTLE_G), at "1. Preliminary Statement" (3b).

In [29]:
# =========================================================
# 4.1 Row roles
# =========================================================
DATE_RE = re.compile(r"^(\d{1,2}\s*[A-Za-z]{3,9},?\s*\d{2,4}|[A-Za-z]{3,9}\.?\s+\d{1,2},?\s*\d{4}|\d{1,2}[A-Za-z]{3}\d{2})\.?$")
ENCLOSURES_RE = re.compile(r"^Encl(?:osures?)?\s*(?:\(\d+\))?\s*:", re.IGNORECASE)   # "Enclosures:", "Encl (1):"
ENCLOSURE_ITEM_RE = re.compile(r"^\(\d+\)\s")
SIGNATURE_OFFICE_RE = re.compile(r"^(By [Dd]irection|Acting\b|Chief\b|Commanding Officer|Officer in Charge|Investigating "
                                 r"Officer|Director\b|Office of|Captain of the Port|Commander\b|Prevention Division|"
                                 r"Deputy\b|Sector\b|Marine Safety|U\.S\. Coast Guard|District\b|Supervisor\b)")
OCR_CAPTION_RE = re.compile(r"^Fig(?:ure|\.)?\s*[lI]\s*[.:\-–—]")         # OCR read "1" as "l" (MERRICK p.6)


def is_caption_start(t: str) -> bool:
    t = t.strip()
    return bool((CAPTION_START_RE.match(t) or OCR_CAPTION_RE.match(t)) and not FIG_REFERENCE_RE.match(t)
                and not CFR_TABLE_RE.match(t))


# reviewed: body sentences wrapped so that a line starts with "Figure N." — the page has no image, the real caption
# of that figure is printed elsewhere (SPIRIT_OF_BOSTON Figure 16 on p.42)
REFERENCE_NOT_CAPTION = {("SPIRIT_OF_BOSTON", 40, "Figure 16. There is no separate slide")}


CFR_TABLE_RE = re.compile(r"^Table\s*\d+\.\d")          # "Table 2.01-7(a)(4), such as …": a CFR table (WAYMON_BOYD)
FIG_REFERENCE_RE = re.compile(r"^Fig(?:ure|\.)?\s*\d+\s*\)")
NAME_RE = re.compile(r"^[A-Z0-9][A-Z.\-’' ]*[A-Z]\.?\)?(?:,? JR\.?)?$")    # "E. B. SAMMS", "MATTHEW J. MESKUN", "D. H. COST"


def upper_share(t: str) -> float:
    letters = [c for c in t if c.isalpha()]
    return sum(c.isupper() for c in letters) / len(letters) if letters else 0.0


fp_by_report = fingerprint_df.set_index("report_id")
role_records = []
for rid, g in rows_df.groupby("report_id", sort=True):
    g = g.sort_values(["page_number", "y0", "x0"])
    role = {}
    body_start = (int(fp_by_report.at[rid, "io_start_page"]), 0.0)
    io_rows = g[g["text"].map(letter_of).eq("io_report")]
    if len(io_rows):
        body_start = (int(io_rows["page_number"].iloc[0]), float(io_rows["y0"].iloc[0]))
    toc_pages = set(fp_by_report.at[rid, "toc_pages"])
    # running header: top-band rows repeated on >= 3 body pages (3.1)
    # compared on their letters only and fuzzily: OCR reads the same header slightly differently on each page
    # ("PROWLER (O.N." / "PROWLER(O.N.", "FOLLOWING MAN" / "FOLLQWING MÁN")
    top = g[(g["y1"] <= TOP_BAND) & (g["page_number"] > body_start[0])]
    clusters: list[tuple[str, set]] = []
    for r in top.itertuples():
        key = re.sub(r"[^A-Z]", "", r.text.upper())
        if len(key) < 4:
            continue
        for ck, pages in clusters:
            if difflib.SequenceMatcher(None, key, ck).ratio() >= 0.85:
                pages.add(r.page_number)
                break
        else:
            clusters.append((key, {r.page_number}))
    running = [ck for ck, pages in clusters if len(pages) >= 3]


    def is_running(t: str) -> bool:
        key = re.sub(r"[^A-Z]", "", t.upper())
        return len(key) >= 4 and any(difflib.SequenceMatcher(None, key, ck).ratio() >= 0.85 for ck in running)


    for r in g.itertuples():
        t = r.text.strip()
        if r.ocr_noise:
            role[r.row_id] = "ocr_noise"
        elif r.figure_internal:
            role[r.row_id] = "figure_internal"
        elif r.y0 > BOTTOM_BAND and PAGE_NUMBER_RE.match(t) and abs(r.centre - 306) < 60:
            role[r.row_id] = "page_number"
        elif r.y0 > BOTTOM_BAND and ENCLOSURE_FOOTER_RE.match(t):
            role[r.row_id] = "enclosure_footer"
        elif not re.search(r"[A-Za-z0-9]", t):
            role[r.row_id] = "no_text"
        elif r.y1 <= TOP_BAND and r.page_number > body_start[0] and is_running(t):
            role[r.row_id] = "running_header"
        elif r.page_number == 1 and fp_by_report.at[rid, "cover"]:
            role[r.row_id] = "cover"
        elif r.page_number in toc_pages:
            role[r.row_id] = "toc"

    # ---- headings (a letter heading may continue on the next row: the district name) ----
    rows_list = [r for r in g.itertuples()]
    for i, r in enumerate(rows_list):
        if r.row_id in role:
            continue
        k = letter_of(r.text)
        if k and (abs(r.centre - 306) < 90 or r.text.strip().endswith(":")) or (k == "io_report" and r.text.strip() == "MEMORANDUM"):
            role[r.row_id] = "letter_heading"
            if k == "endorsement_district" and r.text.strip().endswith(",") and i + 1 < len(rows_list):
                nxt = rows_list[i + 1]
                if upper_share(nxt.text) > 0.9 and nxt.page_number == r.page_number:
                    role[nxt.row_id] = "letter_heading_cont"
        elif (r.page_number, r.y0) >= body_start and r.x0 < 150 and len(r.text) < 90 and section_of(r.text):
            role[r.row_id] = "section_heading"
        elif any(rx.match(" ".join(r.text.split())) for rx in FRONT_RE.values()):
            role[r.row_id] = "front_heading"

    # ---- letter / report starts, anchored on the first heading of a page (robust to OCR: "13732", merged rows) ----
    # above the heading: the capitalised casualty title; above it, at the right, the file block + date; the rest of
    # the top of the page is the letterhead
    for p, pg in g.groupby("page_number"):
        # anchor = the first letter heading; a report printed without heading starts at "1. Preliminary Statement"
        anchors = pg["row_id"].map(role).eq("letter_heading") | (
            ~fp_by_report.at[rid, "report_heading"] & pg["row_id"].map(role).eq("section_heading")
            & pg["text"].map(lambda t: (section_of(t) or (0,))[0] == 1))
        heads = pg[anchors].sort_values("y0")
        if heads.empty:
            continue
        h = heads.iloc[0]
        above = pg[(pg["y0"] < h.y0) & ~pg["row_id"].isin(role)].sort_values("y0", ascending=False)
        in_title = True
        for r in above.itertuples():
            t = r.text.strip()
            if in_title and upper_share(t) >= 0.85 and len(t) >= 3 and r.x0 > 55 and not FILE_BLOCK_RE.match(t) \
                    and not DATE_RE.match(t):
                role[r.row_id] = "casualty_title"
                continue
            in_title = False
            if r.x0 > 280 and (FILE_BLOCK_RE.match(t) or re.match(r"^1\d732\b", t) or re.match(r"^[A-Z]-\d{4,}", t)):
                role[r.row_id] = "file_block"
            elif r.x0 > 280 and DATE_RE.match(t):
                role[r.row_id] = "file_date"
            else:
                role[r.row_id] = "letterhead"
    # continuation pages of a letter repeat the file block + date at the top right
    for p, pg in g.groupby("page_number"):
        for r in pg[(pg["y0"] < 80) & (pg["x0"] > 280) & ~pg["row_id"].isin(role)].itertuples():
            t = r.text.strip()
            if FILE_BLOCK_RE.match(t):
                role[r.row_id] = "file_block"
            elif DATE_RE.match(t):
                role[r.row_id] = "file_date"

    # ---- memorandum header (From/To/Subj/Ref + their continuation rows) ----
    in_memo = False
    for r in rows_list:
        t = r.text.strip()
        if role.get(r.row_id) == "letter_heading" and t == "MEMORANDUM":
            in_memo = True
            continue
        if in_memo:
            if r.row_id in role:
                continue
            if MEMO_HEADER_RE.match(t) or (r.x0 > 95 and not PARA_NUM_RE.match(t) and not re.match(r"^\d+\.\s", t)):
                role[r.row_id] = "memo_header"
                continue
            in_memo = False

    # ---- signature blocks: a rank row (", U.S. Coast Guard") + the name above it + office rows below ----
    for i, r in enumerate(rows_list):
        if r.row_id in role or not RANK_RE.search(r.text) or r.x0 < 120 or len(r.text) > 90:
            continue
        role[r.row_id] = "signature"
        j = i - 1
        while j >= 0 and rows_list[j].page_number == r.page_number and rows_list[j].row_id not in role \
                and rows_list[j].x0 > 120 and r.y0 - rows_list[j].y1 < 30 and NAME_RE.match(rows_list[j].text.strip()):
            role[rows_list[j].row_id] = "signature"
            j -= 1
        j, prev = i + 1, r
        while j < len(rows_list) and rows_list[j].page_number == r.page_number and rows_list[j].row_id not in role \
                and rows_list[j].x0 > 120 and rows_list[j].y0 - prev.y1 < 12:
            role[rows_list[j].row_id] = "signature"
            prev = rows_list[j]
            j += 1

    # ---- enclosures list ----
    for i, r in enumerate(rows_list):
        if r.row_id in role or not ENCLOSURES_RE.match(r.text.strip()) or r.x0 > 150:
            continue
        role[r.row_id] = "enclosures"
        j = i + 1
        while j < len(rows_list) and rows_list[j].page_number == r.page_number and rows_list[j].row_id not in role \
                and ENCLOSURE_ITEM_RE.match(rows_list[j].text.strip()):
            role[rows_list[j].row_id] = "enclosures"
            j += 1

    # ---- captions (+ continuation rows: smaller type / tight spacing, same column) ----
    for i, r in enumerate(rows_list):
        if r.row_id in role or not is_caption_start(r.text) or any(
                (r.report_id, r.page_number) == (a, b) and r.text.strip().startswith(c) for a, b, c in REFERENCE_NOT_CAPTION):
            continue                                    # "Figure 3) was removed …": a wrapped in-text reference (WAYMON_BOYD)
        role[r.row_id] = "caption"
        j, prev = i + 1, r
        while j < len(rows_list):
            n = rows_list[j]
            if n.page_number != r.page_number or n.row_id in role:
                break
            # same type as the caption's first row (native) / not taller (OCR), set tight under it, same column
            same_type = abs(n.size - r.size) <= 0.6 if n.provenance == "native" else n.height <= 1.1 * r.height + 0.5
            tight = n.y0 - prev.y1 < max(4.0, 0.6 * prev.height)
            if not (same_type and tight) or is_caption_start(n.text) or PARA_NUM_RE.match(n.text.strip()):
                break
            role[n.row_id] = "caption"
            prev, j = n, j + 1

    # ---- native text inside an image placement that is not a caption: labels printed on a figure ----
    for r in rows_list:
        if r.row_id not in role and r.in_image:
            role[r.row_id] = "figure_internal"

    # ---- footnotes (native): small type at the bottom of the page, starting with the note number ----
    for p, pg in g.groupby("page_number"):
        cand = pg[(pg["provenance"] == "native") & (pg["size"] <= 10.5) & (pg["y0"] > 480) & ~pg["row_id"].isin(role)]
        started = False
        for r in cand.sort_values("y0").itertuples():
            if re.match(r"^\d{1,3}\s*\S", r.text.strip()) or started:
                role[r.row_id] = "footnote"
                started = True

    for r in rows_list:
        role.setdefault(r.row_id, "body")
    role_records += [{"row_id": k, "role": v} for k, v in role.items()]

roles_df = pd.DataFrame(role_records)
rows_df = rows_df.drop(columns=["role"], errors="ignore").merge(roles_df, on="row_id", how="left")
print(rows_df["role"].value_counts().to_dict())

{'body': 14054, 'letterhead': 627, 'page_number': 503, 'caption': 368, 'casualty_title': 262, 'figure_internal': 230, 'signature': 212, 'section_heading': 182, 'letter_heading': 177, 'cover': 169, 'file_block': 157, 'footnote': 146, 'file_date': 128, 'toc': 100, 'running_header': 56, 'enclosures': 45, 'memo_header': 18, 'no_text': 14, 'letter_heading_cont': 12, 'enclosure_footer': 9, 'front_heading': 1, 'ocr_noise': 1}


In [30]:
# =========================================================
# 4.2 Document parts (letters, report sections), dates, incident date, vessel label
# =========================================================
MONTHS = ["january", "february", "march", "april", "may", "june", "july", "august", "september", "october",
          "november", "december"]


def parse_date(t: str) -> str | None:
    """ISO date of a printed date ('27 March 2026', 'March 16, 2026', '26 Jan 2026', '06May13', '03Mar 2026',
    'October 17,2019'); an OCR-damaged month ('Junc') is matched to the closest month name."""
    t = " ".join(t.replace(",", " , ").split())
    m = re.search(r"(\d{1,2})\s*([A-Za-z]{3,9})\.?\s*,?\s*(\d{4}|\d{2})\b", t) or None
    if m:
        d, mon, y = m.group(1), m.group(2), m.group(3)
    else:
        m = re.search(r"([A-Za-z]{3,9})\.?\s+(\d{1,2})\s*,?\s*(\d{4})\b", t)
        if not m:
            return None
        mon, d, y = m.group(1), m.group(2), m.group(3)
    mon = mon.lower()
    cands = [k for k in MONTHS if k.startswith(mon[:3])] or difflib.get_close_matches(mon, MONTHS, n=1, cutoff=0.6)
    if not cands:
        return None
    year = int(y) + (2000 if len(y) == 2 else 0)
    try:
        return pd.Timestamp(year=year, month=MONTHS.index(cands[0]) + 1, day=int(d)).strftime("%Y-%m-%d")
    except ValueError:
        return None


assert parse_date("27 March 2026") == "2026-03-27" and parse_date("October 17,2019") == "2019-10-17"
assert parse_date("06May13") == "2013-05-06" and parse_date("Junc 5, 2025") == "2025-06-05"
assert parse_date("03Mar 2026") == "2026-03-03" and parse_date("18 Dec, 2025") == "2025-12-18"

PARENT_PARTS = {"action_commandant", "endorsement_district", "endorsement_ocmi", "executive_summary", "io_report"}
SUB_PARTS = {"action_on_recommendations", "endorsement_on_recommendations", "comments_on_report"}

part_records, part_of_row = [], {}
for rid, g in rows_df.sort_values(["report_id", "page_number", "y0", "x0"]).groupby("report_id", sort=True):
    has_report_heading = bool(fp_by_report.at[rid, "report_heading"])
    real_io_seen = any(letter_of(t) == "io_report" and t.strip() != "MEMORANDUM"
                       for t in g.loc[g["role"] == "letter_heading", "text"])
    n, part, section, sub = 0, "cover", None, None
    for r in g.itertuples():
        if r.role == "letter_heading":
            k = letter_of(r.text)
            if r.text.strip() == "MEMORANDUM" and real_io_seen:
                k = "convening_memorandum"          # SPIRIT_OF_BOSTON: the order convening the formal investigation
            if k in PARENT_PARTS or k == "convening_memorandum":
                n += 1
                part, section, sub = k, None, None
                part_records.append({"report_id": rid, "part_no": n, "part": k, "heading_row": r.row_id,
                                     "heading": " ".join(r.text.split()), "page_start": r.page_number})
            elif k in SUB_PARTS:
                sub = k
        elif r.role == "front_heading":
            n += 1
            part = "list_of_acronyms" if FRONT_RE["list_of_acronyms"].match(" ".join(r.text.split())) else "front"
            section, sub = None, None
            part_records.append({"report_id": rid, "part_no": n, "part": part, "heading_row": r.row_id,
                                 "heading": " ".join(r.text.split()), "page_start": r.page_number})
        elif r.role == "section_heading":
            if part != "io_report":                        # CUYAHOGA / LITTLE_G: the report starts at section 1
                n += 1
                part = "io_report"
                part_records.append({"report_id": rid, "part_no": n, "part": "io_report", "heading_row": None,
                                     "heading": "(no heading printed)", "page_start": r.page_number})
            section, sub = section_of(r.text)[0], None
        part_of_row[r.row_id] = (n, part, section, sub)

rows_df["part_no"] = rows_df["row_id"].map(lambda i: part_of_row[i][0])
rows_df["part"] = rows_df["row_id"].map(lambda i: part_of_row[i][1])
rows_df["section"] = rows_df["row_id"].map(lambda i: part_of_row[i][2]).astype("Int64")
rows_df["sub_part"] = rows_df["row_id"].map(lambda i: part_of_row[i][3])
parts_df = pd.DataFrame(part_records)

# a letter's date = the file date printed above its heading, on the heading's page
dates = []
for p in parts_df.itertuples():
    same_page = rows_df[(rows_df["report_id"] == p.report_id) & (rows_df["page_number"] == p.page_start)]
    head_y = rows_df.set_index("row_id").at[p.heading_row, "y0"] if isinstance(p.heading_row, str) else 1e9
    fd = same_page[(same_page["role"] == "file_date") & (same_page["y0"] < head_y)]
    raw = fd["text"].iloc[-1].strip() if len(fd) else None
    dates.append((raw, parse_date(raw) if raw else None))
parts_df["date_printed"] = [d[0] for d in dates]
parts_df["date"] = [d[1] for d in dates]
parts_df["page_end"] = parts_df.groupby("report_id")["page_start"].shift(-1).fillna(
    parts_df["report_id"].map(reports_df.set_index("report_id")["pages"])).astype(int)

# ---- version_date = the Commandant's action (the later one when there are two: QUEEN_OF_THE_WEST) ----
cmd = parts_df[parts_df["part"] == "action_commandant"]
VERSION_DATE_OF = cmd.groupby("report_id")["date"].max().to_dict()

# ---- incident date: the date closing the casualty titles ("… ON APRIL 6, 2024"), most frequent across the letters ----
INCIDENT_DATE_RE = re.compile(r"\b(?:ON|OF)\s+([A-Z][A-Za-z]+\.?\s+\d{1,2},?\s*\d{4})", re.IGNORECASE)
incident = {}
for rid, g in rows_df[rows_df["role"].isin(["casualty_title", "cover"])].groupby("report_id"):
    found = []
    for (_, pn), tg in g.groupby(["part_no", "page_number"]):
        txt = " ".join(tg.sort_values("y0")["text"])
        found += [parse_date(m) for m in INCIDENT_DATE_RE.findall(txt)]
    found = [f for f in found if f]
    incident[rid] = Counter(found).most_common(1)[0][0] if found else None
INCIDENT_DATE_OF = incident

# ---- vessel label for chunk headers: the first official name; both vessels for the collision named in the file ----
OFFICIAL_NAME_RE = re.compile(r"^(?:Official Name|Vessel Name|Name)(?:/State Number)?\s*:\s*(.+)$", re.IGNORECASE)
VESSEL_LABEL_OF = {}
for rid, g in rows_df.groupby("report_id"):
    names = [OFFICIAL_NAME_RE.match(t).group(1).strip() for t in g["text"] if OFFICIAL_NAME_RE.match(t)]
    VESSEL_LABEL_OF[rid] = names[0].upper()
VESSEL_LABEL_OF["PROWLER_ATTESSA_IV"] = "PROWLER / ATTESSA IV"      # collision: both vessels are the subject (file name)

meta_df = pd.DataFrame({"report_id": REPORT_IDS})
meta_df["misle"] = meta_df["report_id"].map(MISLE_OF_FILE)
meta_df["vessel_label"] = meta_df["report_id"].map(VESSEL_LABEL_OF)
meta_df["incident_date"] = meta_df["report_id"].map(INCIDENT_DATE_OF)
meta_df["version_date"] = meta_df["report_id"].map(VERSION_DATE_OF)
print("Report metadata:", meta_df.shape, "| missing values:", meta_df.isna().sum()[lambda s: s > 0].to_dict())
print("Parts per kind:", parts_df.groupby("part").size().to_dict())
display(meta_df.head(5))

Report metadata: (23, 5) | missing values: {}
Parts per kind: {'action_commandant': 24, 'convening_memorandum': 1, 'endorsement_district': 23, 'endorsement_ocmi': 20, 'executive_summary': 22, 'io_report': 25, 'list_of_acronyms': 1}


,report_id,misle,vessel_label,incident_date,version_date
0,BIG_STICK,7895450,BIG STICK,2024-04-06,2026-03-27
1,CAROL_ANN,7816410,CAROL ANN,2023-10-14,2026-01-12
2,CONCHRD,7107811,CONCH’RD,2020-12-03,2026-01-16
3,CUMBERLAND_II,8126294,CUMBERLAND II,2025-04-25,2026-06-23
4,CUYAHOGA,7881082,CUYAHOGA,2024-03-15,2026-08-31


In [31]:
# =========================================================
# 4.3 Tables: vessel / vehicle blocks (Section 2) and persons tables (Section 3)
# =========================================================
# Cells come from the drawn horizontal rules of the table (native: vector drawings; scanned pages: long dark pixel
# runs in the page image), because a label is printed in the vertical middle of a multi-line cell (CUMBERLAND_II
# "Owner:" beside a 4-line address). Without rules (the memorandum), a row without label continues the previous cell.
LABEL_CANON = [
    (r"(Official Name(/State Number)?|Vessel Name|Name)", "name"),
    (r"(Vessel )?Identification Number|Official Number", "identification_number"),
    (r"Hull Identification Number", "hull_identification_number"),
    (r"Flag", "flag"),
    (r"Vessel Class/Type/Sub-?Type|Service", "vessel_class"),
    (r"Build Year", "build_year"),
    (r"Gross Ton(nage|s)( \(GRT\))?", "gross_tonnage"),
    (r"(Registered )?Length", "length"),
    (r"Beam(/Width)?", "beam"),
    (r"Draft(/Depth)?|Depth", "draft_depth"),
    (r"Hull Material|Hull Construction", "hull"),
    (r"Main[/ ]Primary Propulsion.*|Propulsion", "propulsion"),
    (r"Maximum Passengers and Crewpersons", "max_persons"),
    (r"Owner ?/ ?Operator", "owner_operator"),
    (r"Owner(/Managing Owner)?", "owner"),
    (r"Operator", "operator"),
    (r"Make", "vehicle_make"), (r"Model", "vehicle_model"), (r"Year", "vehicle_year"), (r"Color", "vehicle_color"),
    (r"License Plate", "vehicle_license_plate"), (r"State and County of Registration", "vehicle_registration"),
]
LABEL_CANON_RE = [(re.compile(rf"^(?:{p})\s*:?$", re.IGNORECASE), f) for p, f in LABEL_CANON]
LABEL_HEAD_RE = re.compile(r"^(?P<label>[A-Za-z][A-Za-z/()\-, ]{1,45}?)\s*:\s*(?P<rest>.*)$")


def canon_label(label: str) -> str | None:
    lab = re.sub(r"\s*\(.*$", "", " ".join(label.split())).strip(" :")       # "Main/Primary Propulsion (Configuration…"
    return next((f for rx, f in LABEL_CANON_RE if rx.match(lab)), None)


def has_label(first_seg: tuple, split: float) -> bool:
    """True when the row's first segment is a known label left of the value column."""
    x, _, t = first_seg
    m = LABEL_HEAD_RE.match(t)
    return bool(x < split and ((m and canon_label(m["label"])) or canon_label(t)))


def horizontal_rules(rid: str, p: int, x0: float, x1: float) -> list[float]:
    """y positions (pt) of horizontal lines spanning >= 50 % of [x0, x1] on page p."""
    ys = []
    if pages_df.set_index(["report_id", "page_number"]).at[(rid, p), "text_source"] == "native":
        for d in docs[rid][p - 1].get_drawings():
            for it in d["items"]:
                if it[0] == "re":
                    r = it[1]
                    if r.height <= 2.5 and r.width >= 0.5 * (x1 - x0) and r.x0 < x1 and r.x1 > x0:
                        ys.append((r.y0 + r.y1) / 2)
                elif it[0] == "l" and abs(it[1].y - it[2].y) < 1.0 and abs(it[2].x - it[1].x) >= 0.5 * (x1 - x0):
                    ys.append(it[1].y)
    else:
        s = GRAPHIC_DPI / 72
        pix = docs[rid][p - 1].get_pixmap(dpi=GRAPHIC_DPI, colorspace=pymupdf.csGRAY)
        grey = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width)
        band = grey[:, max(0, int(x0 * s)):int(x1 * s)] < 140
        dark = band.sum(axis=1) >= 0.5 * band.shape[1]
        ys = [i / s for i in np.flatnonzero(dark)]
    merged = []
    for y in sorted(ys):
        if not merged or y - merged[-1][-1] > 2.0:
            merged.append([y])
        else:
            merged[-1].append(y)
    return [round(float(np.mean(m)), 1) for m in merged]


def table_rows_of(sec_rows: pd.DataFrame, split: float) -> pd.DataFrame:
    """The key–value rows of a Section 2 part: rows with a label, value-only rows and label continuations."""
    keep, started = [], False
    for r in sec_rows.itertuples():
        first_x, _, first_t = r.segments[0]
        labelled = first_x < split and (LABEL_HEAD_RE.match(first_t) and canon_label(LABEL_HEAD_RE.match(first_t)["label"])
                                        or canon_label(first_t))
        value_only = first_x >= split - 25
        continuation = started and len(r.segments) >= 1 and first_x < split and not labelled \
            and (len(r.segments) > 1 or len(first_t) < 60) and not PARA_NUM_RE.match(first_t)
        if r.role == "body" and (labelled or (started and (value_only or continuation))):
            keep.append(r.row_id)
            started = True
        elif started and r.role not in ("page_number", "running_header", "caption", "figure_internal"):
            started = False
    return sec_rows[sec_rows["row_id"].isin(keep)]


vessel_field_records = []
for rid, g in rows_df[(rows_df["section"] == 2)].groupby("report_id", sort=True):
    g = g.sort_values(["page_number", "y0", "x0"])
    two = [seg for segs in g.loc[g["role"] == "body", "segments"] for seg in segs[1:2]]
    split = float(np.median([s[0] for s in two])) - 10 if two else 250.0
    trows = table_rows_of(g, split)
    cells = []                                          # list of lists of row_ids
    for p, pg in trows.groupby("page_number", sort=True):
        x0, x1 = float(pg["x0"].min()), float(pg["x1"].max())
        rules = horizontal_rules(rid, int(p), x0, max(x1, x0 + 200))
        rules = [y for y in rules if pg["y0"].min() - 20 <= y <= pg["y1"].max() + 20]
        if len(rules) >= 3:     # a rule the page image lost at the top / bottom (H20_BELOW, MERRICK): close the table
            rules = sorted({pg["y0"].min() - 1, *rules, pg["y1"].max() + 1})
        if len(rules) >= 3:
            for a, b in zip(rules, rules[1:]):
                members = pg[((pg["y0"] + pg["y1"]) / 2 > a) & ((pg["y0"] + pg["y1"]) / 2 < b)]
                labelled = [r for r in members.itertuples() if has_label(r.segments[0], split)]
                if len(labelled) >= 2:
                    # a rule the page image lost (OCR'd pages: H20_BELOW Beam/Draft, QUEEN_OF_THE_WEST Gross Tons/Hull):
                    # one sub-cell per labelled row, an unlabelled row joins the nearest one
                    ymid = lambda r: (r.y0 + r.y1) / 2
                    sub = {r.row_id: [] for r in labelled}
                    for r in members.itertuples():
                        sub[min(labelled, key=lambda l: abs(ymid(l) - ymid(r))).row_id].append(r.row_id)
                    cells.extend(sub.values())
                elif len(members):
                    cells.append(members["row_id"].tolist())
        else:
            for r in pg.itertuples():
                if has_label(r.segments[0], split) or not cells:
                    cells.append([r.row_id])
                else:
                    cells[-1].append(r.row_id)
    R = rows_df.set_index("row_id")
    vessel_no, block = 0, "vessel"
    for cell in cells:
        segs = [s for rid_ in cell for s in R.at[rid_, "segments"]]
        label_parts = [t for x, _, t in segs if x < split]
        value_parts = [t for x, _, t in segs if x >= split]
        label = " ".join(label_parts)
        m = LABEL_HEAD_RE.match(label)
        if m and m["rest"] and not value_parts and not m["rest"].startswith("("):   # "Official Name/State Number: LA6893ER"
            label, value_parts = m["label"], [m["rest"]]
        value = " ".join(" ".join(value_parts).split())
        if not label.strip() and vessel_field_records and vessel_field_records[-1]["report_id"] == rid:
            prev = vessel_field_records[-1]                                     # a value-only cell continues the previous one
            prev["value"] = f"{prev['value']} {value}".strip()
            prev["row_ids"] = prev["row_ids"] + cell
            continue
        field = canon_label(label)
        if field == "name":
            vessel_no, block = vessel_no + 1, "vessel"
        if field and field.startswith("vehicle_") and block != "vehicle":
            block, vessel_no = "vehicle", vessel_no + 1
        vessel_field_records.append({
            "report_id": rid, "block_no": max(vessel_no, 1), "block_kind": block,
            "label_printed": " ".join(label.split()).strip(" :"), "field": field, "value": value,
            "page_number": int(R.at[cell[0], "page_number"]), "row_ids": cell})

vessel_fields_df = pd.DataFrame(vessel_field_records)
assert set(r for ids in vessel_fields_df["row_ids"] for r in ids) == set(
    pd.concat([table_rows_of(g.sort_values(["page_number", "y0", "x0"]),
                             float(np.median([s[0] for segs in g.loc[g["role"] == "body", "segments"] for s in segs[1:2]])) - 10)
               for _, g in rows_df[rows_df["section"] == 2].groupby("report_id")])["row_id"]), "a table row fell outside every cell"

# ---- persons tables (Section 3): columns from the header row; a row is a new person when it has a value in the
#      relationship / position column AND in another column, otherwise it continues the previous person's cells ----
PERSONS_SUBHEADER_RE = re.compile(r"^(Deceased|Survivor)\s+Age\s+Position")      # the memorandum's two tables
person_records, persons_statement = [], {}
for rid, g in rows_df[(rows_df["section"] == 3) & (rows_df["role"] == "body")].groupby("report_id", sort=True):
    g = g.sort_values(["page_number", "y0", "x0"])
    cols, table_no, key_col, current = None, 0, None, None
    for r in g.itertuples():
        t = r.text.strip()
        if PERSONS_HEADER_RE.match(t) or PERSONS_SUBHEADER_RE.match(t):
            table_no += 1
            cols = [(a, b, " ".join(h.split())) for a, b, h in r.segments]
            key_col = next((i for i, (_, _, h) in enumerate(cols) if re.match(r"(Relationship|Position)", h, re.I)), 0)
            header_kind = PERSONS_SUBHEADER_RE.match(t).group(1).lower() if PERSONS_SUBHEADER_RE.match(t) else None
            current = None
            continue
        if cols is None:
            persons_statement.setdefault(rid, []).append(t)             # 'None', 'There were no deceased …'
            continue
        cells = {}
        for a, b, txt in r.segments:
            centre = (a + b) / 2
            k = min(range(len(cols)), key=lambda i: 0 if cols[i][0] - 5 <= centre <= cols[i][1] + 5
                    else min(abs(centre - cols[i][0]), abs(centre - cols[i][1])))
            cells[k] = f"{cells.get(k, '')} {txt}".strip()
        if key_col in cells and len(cells) >= 2:
            current = {"report_id": rid, "table_no": table_no, "table_kind": header_kind,
                       "cells": {cols[i][2]: v for i, v in cells.items()}, "row_ids": [r.row_id],
                       "page_number": r.page_number}
            person_records.append(current)
        elif current is not None:
            for i, v in cells.items():
                current["cells"][cols[i][2]] = f"{current['cells'].get(cols[i][2], '')} {v}".strip()
            current["row_ids"].append(r.row_id)
        else:
            persons_statement.setdefault(rid, []).append(t)


def cell_of(cells: dict, pattern: str) -> str | None:
    return next((v for k, v in cells.items() if re.match(pattern, k, re.I)), None)


persons_df = pd.DataFrame([{
    "report_id": p["report_id"], "table_no": p["table_no"], "table_kind": p["table_kind"],
    "relationship": cell_of(p["cells"], r"(Relationship|Position)"),
    "sex": cell_of(p["cells"], r"Sex"), "age": cell_of(p["cells"], r"Age"),
    "status": cell_of(p["cells"], r"Status") or ({"deceased": "Deceased", "survivor": "Survivor"}.get(p["table_kind"])),
    "page_number": p["page_number"], "row_ids": p["row_ids"], "n_cells": len(p["cells"]),
} for p in person_records])
# a table whose only row is 'None N/A …' (CUYAHOGA) records no person
none_rows = persons_df["relationship"].fillna("").str.match(r"^(None|N/A)$") & persons_df["status"].fillna("N/A").eq("N/A")
persons_statement.update({r.report_id: persons_statement.get(r.report_id, []) + ["table printed with 'None'"]
                          for r in persons_df[none_rows].itertuples()})
persons_none_rows = persons_df[none_rows]
persons_df = persons_df[~none_rows].reset_index(drop=True)
print("Persons:", len(persons_df), "in", persons_df["report_id"].nunique(), "reports | reports without persons:",
      sorted(set(REPORT_IDS) - set(persons_df["report_id"])))
display(persons_df[["report_id", "table_no", "relationship", "sex", "age", "status"]].head(5))
print("Vessel / vehicle fields:", len(vessel_fields_df), "| unknown labels:",
      vessel_fields_df.loc[vessel_fields_df["field"].isna(), ["report_id", "label_printed", "value"]].values.tolist()[:10])
print("Blocks per kind:", vessel_fields_df.drop_duplicates(["report_id", "block_no"])["block_kind"].value_counts().to_dict(),
      "| reports:", vessel_fields_df["report_id"].nunique())
display(vessel_fields_df[["report_id", "block_no", "block_kind", "field", "value"]].head(5))

Persons: 41 in 19 reports | reports without persons: ['CUYAHOGA', 'MICHIPICOTEN', 'SPIRIT_OF_BOSTON', 'VICAR_602']


,report_id,table_no,relationship,sex,age,status
0,BIG_STICK,1,Operator,Male,46,Missing and Presumed Deceased
1,CAROL_ANN,1,Master,Male,24,Missing (Presumed Dead)
2,CAROL_ANN,1,Deckhand 1 (DH1),Male,24,Missing (Presumed Dead)
3,CAROL_ANN,1,Deckhand 2 (DH2),Male,27,Missing (Presumed Dead)
4,CONCHRD,1,Crewman,Male,45,Deceased (presumed dead)


Vessel / vehicle fields: 297 | unknown labels: []
Blocks per kind: {'vessel': 26, 'vehicle': 1} | reports: 23


,report_id,block_no,block_kind,field,value
0,BIG_STICK,1,vessel,name,BIG STICK
1,BIG_STICK,1,vessel,identification_number,Official Number 1337362
2,BIG_STICK,1,vessel,flag,United States
3,BIG_STICK,1,vessel,vessel_class,Recreational / General
4,BIG_STICK,1,vessel,build_year,2023


### 4.4–4.7 Persons, figures, enclosures, table of contents, validation

- **Persons (4.4):** the printed relationship/position is the role label (plan §4.6: roles, never names); class
  `crew` / `passenger` / `other` and outcome `dead_or_missing` / `injured` / `uninjured`, plus `survivor_unspecified`
  for the memorandum's survivor table (it does not say whether a survivor was injured, so nothing is guessed).
- **Figures (4.5):** one unit per caption; the image region is recorded so its internal text never leaks into the
  body. Value gate (ADR-020): all figures are raster images with no text-readable data → `caption_only`.
  Wrapped in-text references ("Figure 3) was removed …", "Table 2.01-7(a)(4) …") are not captions.
- **Enclosures and TOC (4.6):** metadata only; they cross-check that the parser found every announced part and section.
- **Validation (4.7):** structural asserts + a reviewed snapshot of every count.

In [32]:
# =========================================================
# 4.4 Persons: role label, class and outcome (plan §4.6: roles, never names)
# =========================================================
# The printed "Relationship to Vessel" / "Position" is the person's role label; the Name column is never kept.
# class: passenger / crew / other (an owner who did not operate the vessel, an industrial worker carried on board);
# outcome from the printed status. "Survivor" (the memorandum's survivor table) does not say whether the person was
# injured, so it gets its own outcome instead of a guessed "uninjured".
PASSENGER_RE = re.compile(r"passenger", re.I)
OTHER_ROLE_RE = re.compile(r"^(Owner|Worker\b.*Industrial)", re.I)          # "Owner" alone; "Owner/Operator" operates
OUTCOME_RULES = [(r"deceased|dead|missing", "dead_or_missing"), (r"^injured", "injured"),
                 (r"^uninjured", "uninjured"), (r"^survivor$", "survivor_unspecified")]


def person_class(rel: str) -> str:
    if PASSENGER_RE.search(rel):
        return "passenger"
    if OTHER_ROLE_RE.match(rel) and not re.search(r"operator", rel, re.I):
        return "other"
    return "crew"


def person_outcome(status: str) -> str | None:
    return next((o for p, o in OUTCOME_RULES if re.search(p, status.strip(), re.I)), None)


persons_df["role_label"] = persons_df["relationship"].map(lambda t: " ".join(t.split()).split(" – ")[0])
persons_df["person_class"] = persons_df["relationship"].map(person_class)
persons_df["outcome"] = persons_df["status"].map(person_outcome)
# SPIRIT_OF_BOSTON prints no persons section (its sections run 1, 2, 4… with the Findings in two parts, 3b)
persons_statement.setdefault("SPIRIT_OF_BOSTON", ["no persons section printed"])
print("Persons:", len(persons_df), "| class:", persons_df["person_class"].value_counts().to_dict(),
      "| outcome:", persons_df["outcome"].value_counts(dropna=False).to_dict())
print("Reports with a persons statement instead of a table:", sorted(persons_statement))
display(persons_df[["report_id", "role_label", "person_class", "outcome"]].head(5))

Persons: 41 | class: {'crew': 34, 'passenger': 5, 'other': 2} | outcome: {'dead_or_missing': 26, 'injured': 8, 'survivor_unspecified': 5, 'uninjured': 2}
Reports with a persons statement instead of a table: ['CUYAHOGA', 'MICHIPICOTEN', 'SPIRIT_OF_BOSTON', 'VICAR_602']


,report_id,role_label,person_class,outcome
0,BIG_STICK,Operator,crew,dead_or_missing
1,CAROL_ANN,Master,crew,dead_or_missing
2,CAROL_ANN,Deckhand 1 (DH1),crew,dead_or_missing
3,CAROL_ANN,Deckhand 2 (DH2),crew,dead_or_missing
4,CONCHRD,Crewman,crew,dead_or_missing


In [33]:
# =========================================================
# 4.5 Figures and tables: caption units, image region, value gate (ADR-020)
# =========================================================
# A caption unit = its start row ("Figure N …", "Table N …", "Above: Photograph …") + the continuation rows Phase 4.1
# gave the role `caption` (credits, second line). Two captions can share a baseline (WAYMON_BOYD p.17): a
# continuation row joins the start row of its own column. The image region is the nearest graphic region of the page.
FIG_NO_RE = re.compile(r"^(?P<kind>Fig(?:ure|\.)?|Table)\s*(?P<no>\d+|[lI](?=\s*[.:\-–—]))", re.I)
DIAGRAM_RE = re.compile(r"\b(chart(let)?|diagram|graph|map|track(line)?|AIS|plot|timeline|schematic|drawing|layout|"
                        r"sketch|arrangement|illustration|rendering|ECDIS|radar|plan view|profile view|screenshot|"
                        r"screen ?shot|data|weather|forecast|buoy)\b", re.I)

cap = rows_df[rows_df["role"] == "caption"].sort_values(["report_id", "page_number", "y0", "x0"])
figure_records = []
for (rid, p), g in cap.groupby(["report_id", "page_number"], sort=True):
    open_ = []                                                     # figure records started on this page
    for r in g.itertuples():
        t = " ".join(r.text.split())
        if is_caption_start(t):
            m = FIG_NO_RE.match(t)
            rec = {"report_id": rid, "page_number": int(p), "kind": "table" if m and m["kind"].lower() == "table" else "figure",
                   "number": int(m["no"].replace("l", "1").replace("I", "1")) if m else None, "caption": t, "row_ids": [r.row_id],
                   "x0": r.x0, "x1": r.x1, "y0": r.y0, "y1": r.y1}
            figure_records.append(rec); open_.append(rec)
        elif open_:
            rec = max(open_, key=lambda f: min(f["x1"], r.x1) - max(f["x0"], r.x0))     # same column
            rec["caption"] += " " + t; rec["row_ids"].append(r.row_id); rec["y1"] = max(rec["y1"], r.y1)
figures_df = pd.DataFrame(figure_records)
figures_df["figure_id"] = [f"{r.report_id}:fig{i:02d}" for i, r in
                           zip(figures_df.groupby("report_id").cumcount() + 1, figures_df.itertuples())]

graphic_regions = graphics_df[graphics_df["kind"].isin(["graphic", "table"])]
by_page = {k: g for k, g in graphic_regions.groupby(["report_id", "page_number"])}


TEXT_SOURCE_OF = pages_df.set_index(["report_id", "page_number"])["text_source"].to_dict()


def region_of(f) -> tuple | None:
    """Native page: the nearest image placement; OCR'd page: the nearest graphic region of Phase 1b."""
    if TEXT_SOURCE_OF[(f.report_id, f.page_number)] == "native":
        boxes = [tuple(i["bbox"]) for i in docs[f.report_id][f.page_number - 1].get_image_info()
                 if (i["bbox"][2] - i["bbox"][0]) * (i["bbox"][3] - i["bbox"][1]) > 2000]     # not a logo speck
    else:
        g = by_page.get((f.report_id, f.page_number))
        boxes = [] if g is None else list(g["bbox"])
    if not boxes:
        return None
    gap = lambda b: max(f.y0 - b[3], b[1] - f.y1, 0)               # vertical distance caption ↔ region
    return tuple(round(float(v), 1) for v in min(boxes, key=gap))


figures_df["region"] = [region_of(f) for f in figures_df.itertuples()]
figures_df["figure_class"] = np.where(figures_df["kind"] == "table", "table",
                                      np.where(figures_df["caption"].map(lambda c: bool(DIAGRAM_RE.search(c))),
                                               "diagram_or_chart", "photo"))
# Value gate (ADR-020). Every figure here is a raster image: no plotted value can be read as text, and the text
# printed inside it is excluded as `figure_internal` (4.1). The caption carries the figure's claim and the numbered
# findings carry the event, so every unit is `caption_only`; the class only records what the caption says it shows.
figures_df["gate"] = "caption_only"
figures_df["gate_reason"] = figures_df["figure_class"].map({
    "photo": "photograph: caption only (ADR-020)",
    "diagram_or_chart": "raster diagram/chart/map: no text-readable data; caption kept, internal text excluded",
    "table": "table image: caption kept, cell text excluded as figure_internal"})
# printed numbering: a number used twice in one report (SPIRIT_OF_BOSTON 2; WAYMON_BOYD: 4, and a second series 6–13)
# is kept as printed; `figure_id` is the unique key
num = figures_df[figures_df["number"].notna()]
FIGURE_NUMBER_REPEATS = {(r, int(n)) for r, n in num.loc[num.duplicated(["report_id", "kind", "number"]),
                                                         ["report_id", "number"]].itertuples(index=False)}
print("Caption units:", len(figures_df), "| class:", figures_df["figure_class"].value_counts().to_dict(),
      "| without region:", int(figures_df["region"].isna().sum()), "| caption rows used:",
      sum(map(len, figures_df["row_ids"])), "of", len(cap))
print("Printed figure numbers used twice:", sorted(FIGURE_NUMBER_REPEATS))
display(figures_df[["figure_id", "page_number", "figure_class", "caption"]].head(5))

Caption units: 204 | class: {'photo': 167, 'diagram_or_chart': 35, 'table': 2} | without region: 0 | caption rows used: 368 of 368
Printed figure numbers used twice: [('SPIRIT_OF_BOSTON', 2), ('WAYMON_BOYD', 4), ('WAYMON_BOYD', 6), ('WAYMON_BOYD', 7), ('WAYMON_BOYD', 10), ('WAYMON_BOYD', 11), ('WAYMON_BOYD', 12), ('WAYMON_BOYD', 13)]


,figure_id,page_number,figure_class,caption
0,BIG_STICK:fig01,7,photo,Figure 1: Photograph of the BIG STICK taken by...
1,BIG_STICK:fig02,10,diagram_or_chart,Figure 2: Screenshot showing the location of B...
2,BIG_STICK:fig03,11,photo,Figure 3: Boarding team from USS SHERMAN condu...
3,BIG_STICK:fig04,12,photo,Figure 4: Photograph of the BIG STICK’s port s...
4,BIG_STICK:fig05,14,photo,Figure 5: Overlay of search patterns conducted...


In [34]:
# =========================================================
# 4.6 "Enclosures" lists and the table of contents, cross-checked against the parts and sections found
# =========================================================
# The enclosures list of a letter names the parts that follow it; the TOC names the sections and sub-sections of the
# report. Both are metadata (never chunked); here they only check that the parser found what the report announces.
ENCLOSURE_PART_RULES = [(r"executive summary", "executive_summary"), (r"investigating officer|report of investigation|"
                         r"formal investigation report", "io_report"), (r"district|commander", "endorsement_district"),
                        (r"officer in charge|ocmi|captain of the port|sector", "endorsement_ocmi")]
enclosure_records = []
for rid, g in rows_df[rows_df["role"] == "enclosures"].sort_values(["page_number", "y0"]).groupby("report_id"):
    lists = []                                          # one list per letter: a header row starts a new one
    for t in g["text"]:
        t = " ".join(t.split())
        if ENCLOSURES_RE.match(t):
            n = re.match(r"^Encl\w*\s*\((\d+)\)", t, re.I)                  # "Encl (1): Report of …" (CUMBERLAND_II)
            lists.append((f"({n[1]}) " if n else "") + ENCLOSURES_RE.sub("", t, count=1).strip())
        else:
            lists[-1] += " " + t
    for list_no, text in enumerate(lists, start=1):
        for n, item in re.findall(r"\((\d+)\)\s*(.*?)(?=\s*\(\d+\)\s|$)", text):
            part = next((pt for pat, pt in ENCLOSURE_PART_RULES if re.search(pat, item, re.I)), None)
            enclosure_records.append({"report_id": rid, "list_no": list_no, "enclosure_no": int(n), "title": item.strip(),
                                      "part": part,
                                      "part_found": part in set(parts_df.loc[parts_df["report_id"] == rid, "part"])})
enclosures_df = pd.DataFrame(enclosure_records)

TOC_ENTRY_RE = re.compile(r"^(?P<no>\d+(?:\.\d+)*)\.?\s+(?P<title>.+?)\s*(?:\.{3,}\s*)?(?P<page>[ivx\d]+)?$", re.I)
toc_records = []
for rid, g in rows_df[rows_df["role"] == "toc"].sort_values(["page_number", "y0"]).groupby("report_id"):
    for t in g["text"]:
        t = " ".join(t.split())
        m = TOC_ENTRY_RE.match(t)
        if m:
            toc_records.append({"report_id": rid, "no": m["no"], "title": re.sub(r"\s*\.{3,}.*$", "", m["title"])})
        elif toc_records and toc_records[-1]["report_id"] == rid and not re.match(r"^Table of Contents$", t, re.I):
            toc_records[-1]["title"] += " " + re.sub(r"\s*\.{3,}.*$", "", t)      # a title wrapped onto the next row
toc_df = pd.DataFrame(toc_records)


def heading_found(r) -> bool:
    """Top level: a section heading with that number; deeper: a row of the report that starts with that number."""
    g = rows_df[rows_df["report_id"] == r.report_id]
    if "." not in r.no:     # a slip (3b) counts under its printed number too: SPIRIT_OF_BOSTON prints its section 4
        found = {int(k) for k in g["section"].dropna().unique()}           # (Findings) in two parts, "3." and "4."
        return r.no in {str(k) for k in found} | {NUMBERING_SLIPS[(r.report_id, k)] for k in found
                                                   if (r.report_id, k) in NUMBERING_SLIPS}
    return bool(g["text"].map(lambda t: bool(re.match(rf"^{re.escape(r.no)}\.?\s", t.strip()))).any())


toc_df["found"] = [heading_found(r) for r in toc_df.itertuples()]
# reviewed: VICAR_602's letters list an Executive Summary that the public release does not contain (no row of the
# report carries it outside the two lists) — recorded, nothing to parse
ENCLOSURES_NOT_IN_RELEASE = {("VICAR_602", "executive_summary")}
enclosures_df["not_in_release"] = [(r.report_id, r.part) in ENCLOSURES_NOT_IN_RELEASE for r in enclosures_df.itertuples()]
print("Enclosure items:", len(enclosures_df), "in", enclosures_df["report_id"].nunique(), "reports | part found:",
      int(enclosures_df["part_found"].sum()), "| listed, not in the release:", int(enclosures_df["not_in_release"].sum()),
      "| unexplained:", enclosures_df.loc[~enclosures_df["part_found"] & ~enclosures_df["not_in_release"],
                                           ["report_id", "enclosure_no", "title"]].values.tolist()[:8])
print("TOC entries:", len(toc_df), "in", toc_df["report_id"].nunique(), "reports | not found:",
      toc_df.loc[~toc_df["found"], ["report_id", "no", "title"]].values.tolist()[:8])

Enclosure items: 44 in 15 reports | part found: 42 | listed, not in the release: 2 | unexplained: []
TOC entries: 82 in 4 reports | not found: []


In [35]:
# =========================================================
# 4.7 Phase 4 validation
# =========================================================
assert rows_df["role"].notna().all(), "Every row has a role."
assert "name" not in " ".join(persons_df.columns).lower(), "The persons units never keep a Name column (plan §4.6)."
assert meta_df[["misle", "vessel_label", "incident_date", "version_date"]].notna().all().all(), meta_df.isna().sum().to_dict()
# vessels: every report has a named block, every printed label is mapped
assert vessel_fields_df["field"].notna().all(), vessel_fields_df[vessel_fields_df["field"].isna()].head()
assert set(vessel_fields_df.loc[vessel_fields_df["field"] == "name", "report_id"]) == set(REPORT_IDS)
# persons: every report has persons rows or a printed statement, each person has a role, a class and an outcome
assert set(persons_df["report_id"]) | set(persons_statement) == set(REPORT_IDS)
assert persons_df[["role_label", "person_class", "outcome"]].notna().all().all()
# figures: every caption row is in exactly one unit, every unit passed the gate
assert sorted(r for ids in figures_df["row_ids"] for r in ids) == sorted(rows_df.loc[rows_df["role"] == "caption", "row_id"])
assert figures_df["figure_id"].is_unique and figures_df["gate"].notna().all()
# enclosures and TOC agree with what the parser found
assert (enclosures_df["part_found"] | enclosures_df["not_in_release"]).all()
assert toc_df["found"].all(), toc_df[~toc_df["found"]]
# ---- reviewed snapshot (2026-09-26) ----
REVIEWED_PHASE4 = {
    "roles": {"body": 14054, "letterhead": 627, "page_number": 503, "caption": 368, "casualty_title": 262,
        "figure_internal": 230, "signature": 212, "section_heading": 182, "letter_heading": 177, "cover": 169,
        "file_block": 157, "footnote": 146, "file_date": 128, "toc": 100, "running_header": 56, "enclosures": 45,
        "memo_header": 18, "no_text": 14, "letter_heading_cont": 12, "enclosure_footer": 9, "front_heading": 1,
        "ocr_noise": 1},
    "parts": {"io_report": 25, "action_commandant": 24, "endorsement_district": 23, "executive_summary": 22,
        "endorsement_ocmi": 20, "list_of_acronyms": 1, "convening_memorandum": 1},
    "vessel_fields": 297,
    "blocks": {"vessel": 26, "vehicle": 1},
    "persons": 41,
    "person_class": {"crew": 34, "passenger": 5, "other": 2},
    "outcome": {"dead_or_missing": 26, "injured": 8, "survivor_unspecified": 5, "uninjured": 2},
    "figure_class": {"photo": 167, "diagram_or_chart": 35, "table": 2},
    "figures_without_region": [],
    "enclosures": 44,
    "toc": 82}
actual = {"roles": rows_df["role"].value_counts().to_dict(), "parts": parts_df["part"].value_counts().to_dict(),
          "vessel_fields": len(vessel_fields_df),
          "blocks": vessel_fields_df.drop_duplicates(["report_id", "block_no"])["block_kind"].value_counts().to_dict(),
          "persons": len(persons_df), "person_class": persons_df["person_class"].value_counts().to_dict(),
          "outcome": persons_df["outcome"].value_counts().to_dict(),
          "figure_class": figures_df["figure_class"].value_counts().to_dict(),
          "figures_without_region": sorted(figures_df.loc[figures_df["region"].isna(), "figure_id"]),
          "enclosures": len(enclosures_df), "toc": len(toc_df)}
# reviewed: parts printed twice are genuine — CONCHRD's report title page (p.9) + body (p.11), H20_BELOW's two district
# endorsements (Heartland, Eighth), QUEEN_OF_THE_WEST's two Commandant actions and its memorandum + report body
dup_parts = parts_df.groupby(["report_id", "part"]).size()
assert set(dup_parts[dup_parts > 1].index) == {("CONCHRD", "io_report"), ("H20_BELOW", "endorsement_district"),
                                               ("QUEEN_OF_THE_WEST", "action_commandant"),
                                               ("QUEEN_OF_THE_WEST", "io_report")}, dup_parts[dup_parts > 1]
assert FIGURE_NUMBER_REPEATS == {("SPIRIT_OF_BOSTON", 2), *(("WAYMON_BOYD", n) for n in (4, 6, 7, 10, 11, 12, 13))}
assert actual == REVIEWED_PHASE4,{k: v for k, v in actual.items() if v != REVIEWED_PHASE4[k]}
print("✅ Phase 4 validation passed.")
print("Phase 4:", len(meta_df), "reports |", len(parts_df), "parts |", actual["vessel_fields"], "vessel/vehicle fields |",
      actual["persons"], "persons |", len(figures_df), "caption units |", actual["enclosures"], "enclosure items |",
      actual["toc"], "TOC entries")

✅ Phase 4 validation passed.
Phase 4: 23 reports | 116 parts | 297 vessel/vehicle fields | 41 persons | 204 caption units | 44 enclosure items | 82 TOC entries
